In [1]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/Users/jamesjr/Documents/SemiconductorPatentRAG"
)

CORPUS_ROOT = PROJECT_ROOT / "semiconductor_patents"

RAW_ROOT = CORPUS_ROOT / "raw"
CANONICAL_ROOT = CORPUS_ROOT / "canonical"

RAW_ROOT.mkdir(parents=True, exist_ok=True)
CANONICAL_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("SEMICONDUCTOR PATENT CORPUS")
print("=" * 80)

print("Project:")
print(PROJECT_ROOT)

print("\nCorpus:")
print(CORPUS_ROOT)

print("\nRaw:")
print(RAW_ROOT)

print("\nCanonical:")
print(CANONICAL_ROOT)

print("\nExisting files:")
for path in sorted(CORPUS_ROOT.iterdir()):
    print(" ", path.name)

SEMICONDUCTOR PATENT CORPUS
Project:
/Users/jamesjr/Documents/SemiconductorPatentRAG

Corpus:
/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents

Raw:
/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents/raw

Canonical:
/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents/canonical

Existing files:
  01_manufacturing
  02_transistors_devices
  03_memory
  04_advanced_packaging
  05_power_semiconductors
  06_photonics
  07_ai_semiconductors
  advanced_packaging
  ai_advanced_semiconductor
  ai_advanced_semiconductors
  canonical
  csv
  memory
  metadata.csv
  photonics
  power_semiconductors
  raw
  semiconductor_manufacturing
  transistor_device_technology


In [2]:
from pathlib import Path

CORPUS_ROOT = Path(
    "/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents"
)

print("=" * 80)
print("CURRENT CORPUS INVENTORY")
print("=" * 80)

for directory in sorted(CORPUS_ROOT.iterdir()):
    if not directory.is_dir():
        continue

    files = [
        p for p in directory.rglob("*")
        if p.is_file()
    ]

    print(f"\n{directory.name}")
    print(f"  Files: {len(files):,}")

    if files:
        for f in files[:3]:
            print(f"    {f.relative_to(directory)}")

        if len(files) > 3:
            print(f"    ... ({len(files)-3:,} more)")

CURRENT CORPUS INVENTORY

01_manufacturing
  Files: 0

02_transistors_devices
  Files: 0

03_memory
  Files: 0

04_advanced_packaging
  Files: 0

05_power_semiconductors
  Files: 0

06_photonics
  Files: 0

07_ai_semiconductors
  Files: 0

advanced_packaging
  Files: 75
    patent_23ebeda5c273212c3bcd.txt
    patent_d6b919ccb69b6b48ba28.txt
    patent_77ef8db797ce48c60943.txt
    ... (72 more)

ai_advanced_semiconductor
  Files: 50
    patent_974383439b73ac49.txt
    patent_21481e06bf68d6b0.txt
    patent_6294fc102d314418.txt
    ... (47 more)

ai_advanced_semiconductors
  Files: 0

canonical
  Files: 0

csv
  Files: 0

memory
  Files: 75
    patent_2a65b7f115a263fa.txt
    patent_407e4f14a64cafca.txt
    patent_ac5cdd791fc4f412.txt
    ... (72 more)

photonics
  Files: 50
    patent_47dd6eb7e6eae8ab.txt
    patent_eb54814596981097.txt
    patent_98cb2fd1b732cc19.txt
    ... (47 more)

power_semiconductors
  Files: 50
    patent_f4f37be1a1dcf4c4.txt
    patent_acef934600e2200be285.txt


In [3]:
from pathlib import Path

CORPUS_ROOT = Path(
    "/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents"
)

samples = [
    CORPUS_ROOT / "semiconductor_manufacturing" / "patent_5ad88c636ecc0add.txt",
    CORPUS_ROOT / "transistor_device_technology" / "patent_f25e95d6e030dc83.txt",
]

for path in samples:
    print("\n" + "=" * 100)
    print(path)
    print("=" * 100)

    text = path.read_text(errors="ignore")

    print(text[:5000])


/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents/semiconductor_manufacturing/patent_5ad88c636ecc0add.txt
PATENT DOCUMENT
DOCUMENT_ID: 5ad88c636ecc0add
CATEGORY: semiconductor_manufacturing
DATASET: BIGPATENT

ABSTRACT
Method, components, design and fabrication process of a advanced X-ray flat panel detector (FPD), with built-in anti-scattering grid to reduce the X-ray scattering are disclosed. We further disclose two methods in the new X-ray detector: In the first method, the grid is placed on top of X-ray scintillator layer of a FPD, the pixels of X-ray FPD underneath are aligned with the hole structures of anti-scatter grids. The high performance anti-scatter grid applied and aligned to the flat panel detector (FPD) pixel-by-pixel can significantly reduce the noise from the scattered X-rays. The key advantages of the improved art are substantial reduction of grid shadow, improved image contrast-to-noise ratio (CNR) and minimized attenuation of direct X-rays. The

In [4]:
from pathlib import Path
import re
import pandas as pd

CORPUS_ROOT = Path(
    "/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents"
)

# Only the seven actual category directories
CATEGORIES = [
    "semiconductor_manufacturing",
    "transistor_device_technology",
    "memory",
    "advanced_packaging",
    "power_semiconductors",
    "photonics",
    "ai_advanced_semiconductor",
]

records = []

for category in CATEGORIES:
    category_dir = CORPUS_ROOT / category

    for path in sorted(category_dir.glob("*.txt")):
        text = path.read_text(encoding="utf-8", errors="replace")

        # Extract sections
        abstract_match = re.search(
            r"\bABSTRACT\s*\n=+\s*\n(.*?)(?=\nDESCRIPTION\s*\n=+)",
            text,
            re.DOTALL | re.IGNORECASE,
        )

        description_match = re.search(
            r"\bDESCRIPTION\s*\n=+\s*\n(.*)",
            text,
            re.DOTALL | re.IGNORECASE,
        )

        document_id = re.search(
            r"DOCUMENT_ID:\s*(\S+)",
            text,
            re.IGNORECASE,
        )

        abstract = abstract_match.group(1).strip() if abstract_match else ""
        description = description_match.group(1).strip() if description_match else ""

        records.append({
            "category": category,
            "filename": path.name,
            "document_id": document_id.group(1) if document_id else "",
            "abstract_present": bool(abstract),
            "description_present": bool(description),
            "abstract_words": len(abstract.split()),
            "description_words": len(description.split()),
            "total_words": len(text.split()),
            "paragraph_count": len(
                re.findall(r"(?m)^\[\d{4}\]", description)
            ),
        })

audit = pd.DataFrame(records)

print("=" * 70)
print("CORPUS AUDIT")
print("=" * 70)

print(f"\nTotal files: {len(audit)}")

print("\nCategory counts:")
print(audit["category"].value_counts().sort_index())

print("\nMissing abstracts:", (~audit["abstract_present"]).sum())
print("Missing descriptions:", (~audit["description_present"]).sum())
print("Missing document IDs:", (audit["document_id"] == "").sum())

print("\nDuplicate document IDs:",
      audit["document_id"].duplicated().sum())

print("\nDuplicate filenames:",
      audit["filename"].duplicated().sum())

print("\nParagraph statistics:")
print(audit["paragraph_count"].describe())

print("\nWord-count statistics:")
print(audit[["abstract_words", "description_words", "total_words"]].describe())

print("\nFiles with no paragraph markers:")
print((audit["paragraph_count"] == 0).sum())

print("\nFiles with very short descriptions (<100 words):")
print((audit["description_words"] < 100).sum())

print("\nFiles with potential problems:")
problems = audit[
    (~audit["abstract_present"]) |
    (~audit["description_present"]) |
    (audit["document_id"] == "") |
    (audit["paragraph_count"] == 0)
]

print(problems[
    [
        "category",
        "filename",
        "document_id",
        "abstract_present",
        "description_present",
        "paragraph_count",
        "description_words",
    ]
].to_string(index=False))

CORPUS AUDIT

Total files: 500

Category counts:
category
advanced_packaging               75
ai_advanced_semiconductor        50
memory                           75
photonics                        50
power_semiconductors             50
semiconductor_manufacturing     100
transistor_device_technology    100
Name: count, dtype: int64

Missing abstracts: 69
Missing descriptions: 0
Missing document IDs: 69

Duplicate document IDs: 68

Duplicate filenames: 0

Paragraph statistics:
count    500.000000
mean       0.034000
std        0.181411
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: paragraph_count, dtype: float64

Word-count statistics:
       abstract_words  description_words   total_words
count      500.000000         500.000000    500.000000
mean        97.936000        5325.346000   5451.196000
std         56.146315        3361.827365   3368.696184
min          0.000000        1116.000000   1185.000000
25%         64.00000

In [5]:
missing_abstracts = audit[
    ~audit["abstract_present"]
]

print(missing_abstracts[
    ["category", "filename", "document_id"]
].to_string(index=False))

            category                        filename document_id
  advanced_packaging patent_01ea3f6e9244856597cb.txt            
  advanced_packaging patent_03ca7ae1cca3f2ec2c26.txt            
  advanced_packaging patent_04dc1c40cf1047041bea.txt            
  advanced_packaging patent_0628719951bb1c732427.txt            
  advanced_packaging patent_0b0ff476d868d28faab9.txt            
  advanced_packaging patent_0c30900aee0b5eb4018e.txt            
  advanced_packaging patent_0ee4a26e02799ad94449.txt            
  advanced_packaging patent_11f96ffbb8a4144c74fd.txt            
  advanced_packaging patent_1d665cb5eebdc0f2cea0.txt            
  advanced_packaging patent_23ebeda5c273212c3bcd.txt            
  advanced_packaging patent_24a95eaf36a613e9a7ec.txt            
  advanced_packaging patent_26ba5ba2e4e5840b16e8.txt            
  advanced_packaging patent_2d43ab40fc0eb9178499.txt            
  advanced_packaging patent_313810c0b6e6e16b1b9b.txt            
  advanced_packaging pate

In [6]:
print("\nActual duplicate non-empty IDs:")

dupes = audit[
    (audit["document_id"] != "") &
    audit["document_id"].duplicated(keep=False)
]

print(dupes[
    ["category", "filename", "document_id"]
].sort_values("document_id").to_string(index=False))


Actual duplicate non-empty IDs:
Empty DataFrame
Columns: [category, filename, document_id]
Index: []


In [7]:
path = CORPUS_ROOT / "advanced_packaging" / "patent_01ea3f6e9244856597cb.txt"

text = path.read_text(encoding="utf-8", errors="replace")

print(text[:5000])

PATENT DOCUMENT

DOCUMENT ID: 01ea3f6e9244856597cb
CPC SECTION: H
CATEGORY: advanced_packaging
SOURCE: BIGPATENT

ABSTRACT

Embodiments of the invention include a semiconductor integrated circuit package that includes a substrate having an integrated circuit die attached thereto. The package includes a ESD shield attached to the substrate. The ESD shield configured to increase the ESD hardness of the package. The ESD shield can further serve to stiffen the package to prevent warping and operate as a heat spreader.

DETAILED DESCRIPTION

TECHNICAL FIELD 
 The invention described herein relates generally to semiconductor device packaging. In particular, the invention relates to a method and package for a semiconductor device that is resistant to electrostatic discharge that can be caused by various factors. 
 BACKGROUND 
 The semiconductor industry makes wide use of standard BGA (ball grid array) type semiconductor packages. Such packages generally include a BT (bismaleimide triazine) co

In [8]:
from pathlib import Path
import re
import pandas as pd

records = []

for category in CATEGORIES:
    category_dir = CORPUS_ROOT / category

    for path in sorted(category_dir.glob("*.txt")):
        text = path.read_text(encoding="utf-8", errors="replace")

        # Handle both BIGPATENT formats:
        # DOCUMENT_ID: xxx
        # DOCUMENT ID: xxx
        m = re.search(r"DOCUMENT[_ ]ID:\s*(\S+)", text, re.IGNORECASE)
        document_id = m.group(1).strip() if m else path.stem.replace("patent_", "")

        # CPC
        m = re.search(r"CPC(?:[_ ]SECTION)?:\s*(\S+)", text, re.IGNORECASE)
        cpc_section = m.group(1).strip() if m else ""

        # Source
        m = re.search(r"(?:DATASET|SOURCE):\s*(\S+)", text, re.IGNORECASE)
        source = m.group(1).strip() if m else "BIGPATENT"

        # Abstract
        abstract_match = re.search(
            r"ABSTRACT\s*\n=+\s*\n(.*?)(?=\n(?:DESCRIPTION|DETAILED DESCRIPTION)\s*\n=+)",
            text,
            re.DOTALL | re.IGNORECASE,
        )

        abstract = (
            abstract_match.group(1).strip()
            if abstract_match
            else ""
        )

        # Description
        description_match = re.search(
            r"(?:DESCRIPTION|DETAILED DESCRIPTION)\s*\n=+\s*\n(.*)",
            text,
            re.DOTALL | re.IGNORECASE,
        )

        description = (
            description_match.group(1).strip()
            if description_match
            else ""
        )

        records.append({
            "document_id": document_id,
            "category": category,
            "cpc_section": cpc_section,
            "source": source,
            "abstract": abstract,
            "description": description,
            "abstract_words": len(abstract.split()),
            "description_words": len(description.split()),
        })

normalized = pd.DataFrame(records)

print("=" * 70)
print("NORMALIZED CORPUS")
print("=" * 70)

print(f"\nRecords: {len(normalized)}")

print("\nMissing:")
print(normalized[[
    "document_id",
    "abstract",
    "description",
    "cpc_section"
]].isna().sum())

print("\nEmpty:")
print("Document IDs:", (normalized["document_id"] == "").sum())
print("Abstracts:", (normalized["abstract"] == "").sum())
print("Descriptions:", (normalized["description"] == "").sum())
print("CPC sections:", (normalized["cpc_section"] == "").sum())

print("\nDuplicate document IDs:")
print(
    normalized["document_id"].duplicated().sum()
)

print("\nCategory counts:")
print(normalized["category"].value_counts().sort_index())

NORMALIZED CORPUS

Records: 500

Missing:
document_id    0
abstract       0
description    0
cpc_section    0
dtype: int64

Empty:
Document IDs: 0
Abstracts: 0
Descriptions: 0
CPC sections: 431

Duplicate document IDs:
0

Category counts:
category
advanced_packaging               75
ai_advanced_semiconductor        50
memory                           75
photonics                        50
power_semiconductors             50
semiconductor_manufacturing     100
transistor_device_technology    100
Name: count, dtype: int64


In [123]:
NORMALIZED_FILE = CORPUS_ROOT / "normalized_patents.csv"

normalized.to_csv(
    NORMALIZED_FILE,
    index=False
)

print(f"Saved normalized corpus to:")
print(NORMALIZED_FILE)
print(f"Records saved: {len(normalized)}")

Saved normalized corpus to:
/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents/normalized_patents.csv
Records saved: 500


In [124]:
print("\nSaved file exists:", NORMALIZED_FILE.exists())
print("File size:", round(NORMALIZED_FILE.stat().st_size / 1024**2, 2), "MB")


Saved file exists: True
File size: 15.59 MB


In [10]:
# STEP 2 — Inspect normalized patent corpus structure

import re
from collections import Counter

# ---------------------------------------------------------
# 1. BASIC DATASET INSPECTION
# ---------------------------------------------------------

print("=" * 100)
print("NORMALIZED DATASET")
print("=" * 100)

print("Shape:", normalized.shape)

print("\nColumns:")
print(normalized.columns.tolist())

print("\nData types:")
print(normalized.dtypes)

print("\nDescription word-count distribution:")
print(normalized["description_words"].describe())

print("\nAbstract word-count distribution:")
print(normalized["abstract_words"].describe())

print("\nCPC sections:")
print(normalized["cpc_section"].value_counts())

print("\nCategories:")
print(normalized["category"].value_counts())


# ---------------------------------------------------------
# 2. INSPECT REAL DOCUMENTS
# ---------------------------------------------------------

print("\n\n" + "=" * 100)
print("SAMPLE DOCUMENTS")
print("=" * 100)

sample_indices = [0, 100, 300]

for i in sample_indices:
    row = normalized.iloc[i]

    print("\n" + "=" * 100)
    print("DOCUMENT:", row["document_id"])
    print("CATEGORY:", row["category"])
    print("CPC:", row["cpc_section"])
    print("SOURCE:", row["source"])

    print("\nABSTRACT:")
    print(row["abstract"][:1500])

    print("\nDESCRIPTION START:")
    print(row["description"][:4000])


# ---------------------------------------------------------
# 3. DETECT APPARENT SECTION HEADINGS
# ---------------------------------------------------------

print("\n\n" + "=" * 100)
print("APPARENT SECTION HEADINGS")
print("=" * 100)

headings = Counter()

for text in normalized["description"]:
    matches = re.findall(
        r"(?im)^\s*([A-Z][A-Z0-9 ,/&()\-]{3,80})\s*$",
        text
    )

    for heading in matches:
        headings[heading.strip()] += 1

print("\nMost common apparent section headings:")

for heading, count in headings.most_common(30):
    print(f"{count:4}  {heading}")


# ---------------------------------------------------------
# 4. CHECK FOR PARAGRAPH MARKERS
# ---------------------------------------------------------

paragraph_marker_count = 0
documents_with_markers = 0

for text in normalized["description"]:
    matches = re.findall(r"\[\d{4}\]", text)

    if matches:
        documents_with_markers += 1
        paragraph_marker_count += len(matches)

print("\n\n" + "=" * 100)
print("PARAGRAPH MARKER CHECK")
print("=" * 100)

print("Documents with [0000]-style markers:", documents_with_markers)
print("Total paragraph markers:", paragraph_marker_count)


# ---------------------------------------------------------
# 5. IDENTIFY VERY LONG DOCUMENTS
# ---------------------------------------------------------

print("\n\n" + "=" * 100)
print("LONGEST DOCUMENTS")
print("=" * 100)

longest = normalized.nlargest(10, "description_words")[
    [
        "document_id",
        "category",
        "description_words"
    ]
]

print(longest.to_string(index=False))


# ---------------------------------------------------------
# 6. FINAL SUMMARY
# ---------------------------------------------------------

print("\n\n" + "=" * 100)
print("INSPECTION COMPLETE")
print("=" * 100)

print(f"Documents: {len(normalized)}")
print(f"Categories: {normalized['category'].nunique()}")
print(f"CPC sections: {normalized['cpc_section'].nunique()}")
print(
    f"Average description length: "
    f"{normalized['description_words'].mean():.0f} words"
)
print(
    f"Median description length: "
    f"{normalized['description_words'].median():.0f} words"
)
print(f"Documents with paragraph markers: {documents_with_markers}")
print(f"Detected heading patterns: {len(headings)}")

NORMALIZED DATASET
Shape: (500, 8)

Columns:
['document_id', 'category', 'cpc_section', 'source', 'abstract', 'description', 'abstract_words', 'description_words']

Data types:
document_id          object
category             object
cpc_section          object
source               object
abstract             object
description          object
abstract_words        int64
description_words     int64
dtype: object

Description word-count distribution:
count      500.000000
mean      5325.346000
std       3361.827365
min       1116.000000
25%       3342.500000
50%       4637.500000
75%       6309.750000
max      36025.000000
Name: description_words, dtype: float64

Abstract word-count distribution:
count    500.000000
mean     113.022000
std       42.585309
min       28.000000
25%       81.750000
50%      110.000000
75%      141.000000
max      266.000000
Name: abstract_words, dtype: float64

CPC sections:
cpc_section
     431
H     69
Name: count, dtype: int64

Categories:
category
semico

In [11]:
# STEP 2 — Build RAG chunks

import re
import pandas as pd

CHUNK_SIZE = 500
CHUNK_OVERLAP = 75


# ---------------------------------------------------------
# 1. SECTION DETECTION
# ---------------------------------------------------------

HEADING_PATTERN = re.compile(
    r"(?im)^\s*([A-Z][A-Z0-9 ,/&()\-]{3,80})\s*$"
)

IGNORED_HEADINGS = {
    "ABSTRACT",
    "DESCRIPTION",
    "DETAILED DESCRIPTION",
}


def split_into_sections(text):

    matches = list(HEADING_PATTERN.finditer(text))

    if not matches:
        return [("DESCRIPTION", text.strip())]

    sections = []

    # Text before the first heading
    if matches[0].start() > 0:
        preamble = text[:matches[0].start()].strip()

        if preamble:
            sections.append(("DESCRIPTION", preamble))

    for i, match in enumerate(matches):

        heading = match.group(1).strip()

        if heading in IGNORED_HEADINGS:
            continue

        start = match.end()
        end = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(text)
        )

        section_text = text[start:end].strip()

        if section_text:
            sections.append((heading, section_text))

    if not sections:
        return [("DESCRIPTION", text.strip())]

    return sections


# ---------------------------------------------------------
# 2. CHUNK LONG SECTIONS
# ---------------------------------------------------------

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):

    words = text.split()

    if len(words) <= chunk_size:
        return [text.strip()]

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):

        end = start + chunk_size

        chunk = " ".join(words[start:end]).strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(words):
            break

    return chunks


# ---------------------------------------------------------
# 3. CREATE CHUNKS
# ---------------------------------------------------------

chunks = []

for _, row in normalized.iterrows():

    document_id = row["document_id"]
    abstract = row["abstract"].strip()

    # -----------------------------------------------------
    # ABSTRACT CHUNK
    # -----------------------------------------------------

    if abstract:

        chunks.append({
            "document_id": document_id,
            "category": row["category"],
            "cpc_section": row["cpc_section"],
            "source": row["source"],
            "section": "ABSTRACT",
            "chunk_index": 0,
            "abstract": abstract,
            "text": abstract,
            "word_count": len(abstract.split()),
        })

    # -----------------------------------------------------
    # DESCRIPTION CHUNKS
    # -----------------------------------------------------

    sections = split_into_sections(row["description"])

    chunk_index = 1

    for section_name, section_text in sections:

        section_chunks = chunk_text(section_text)

        for part_number, chunk in enumerate(section_chunks, start=1):

            chunks.append({
                "document_id": document_id,
                "category": row["category"],
                "cpc_section": row["cpc_section"],
                "source": row["source"],
                "section": section_name,
                "chunk_index": chunk_index,
                "part": part_number,
                "abstract": abstract,
                "text": chunk,
                "word_count": len(chunk.split()),
            })

            chunk_index += 1


# ---------------------------------------------------------
# 4. DATAFRAME
# ---------------------------------------------------------

chunks_df = pd.DataFrame(chunks)

chunks_df["chunk_id"] = (
    chunks_df["document_id"].astype(str)
    + "_"
    + chunks_df["chunk_index"].astype(str)
)


# ---------------------------------------------------------
# 5. SANITY CHECK
# ---------------------------------------------------------

print("Chunking complete")
print("-" * 50)
print("Documents:", normalized["document_id"].nunique())
print("Total chunks:", len(chunks_df))
print(
    "Average chunks/document:",
    round(len(chunks_df) / normalized["document_id"].nunique(), 1)
)
print(
    "Average words/chunk:",
    round(chunks_df["word_count"].mean(), 1)
)

print("\nChunk types:")
print(chunks_df["section"].value_counts().head(10))

print("\nColumns:")
print(chunks_df.columns.tolist())

print("\nSample:")
print(
    chunks_df[
        [
            "document_id",
            "category",
            "section",
            "word_count"
        ]
    ].head(5).to_string(index=False)
)

Chunking complete
--------------------------------------------------
Documents: 500
Total chunks: 6964
Average chunks/document: 13.9
Average words/chunk: 442.6

Chunk types:
section
DESCRIPTION                                          5641
ABSTRACT                                              500
DETAILED DESCRIPTION OF THE INVENTION                 144
BACKGROUND OF THE INVENTION                            84
SUMMARY OF THE INVENTION                               66
BRIEF DESCRIPTION OF THE DRAWINGS                      64
DETAILED DESCRIPTION OF THE PREFERRED EMBODIMENT       46
DETAILED DESCRIPTION OF THE PREFERRED EMBODIMENTS      39
BACKGROUND                                             29
DETAILED DESCRIPTION OF PREFERRED EMBODIMENTS          24
Name: count, dtype: int64

Columns:
['document_id', 'category', 'cpc_section', 'source', 'section', 'chunk_index', 'abstract', 'text', 'word_count', 'part', 'chunk_id']

Sample:
     document_id                    category     section  wo

In [12]:
# STEP 3A — Test embedding model

from sentence_transformers import SentenceTransformer
import numpy as np
import torch

MODEL_NAME = "BAAI/bge-base-en-v1.5"

# Detect available device
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("Device:", device)
print("Loading:", MODEL_NAME)

model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

# Test with 10 real patent chunks
test_texts = chunks_df["text"].head(10).tolist()

test_embeddings = model.encode(
    test_texts,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("\nEmbedding test passed")
print("Number of embeddings:", len(test_embeddings))
print("Embedding dimension:", test_embeddings.shape[1])
print("Data type:", test_embeddings.dtype)
print("Shape:", test_embeddings.shape)
print(
    "Norm of first embedding:",
    round(float(np.linalg.norm(test_embeddings[0])), 4)
)

/Users/jamesjr/anaconda3/envs/Genomics/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Device: mps
Loading: BAAI/bge-base-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding test passed
Number of embeddings: 10
Embedding dimension: 768
Data type: float32
Shape: (10, 768)
Norm of first embedding: 1.0


In [13]:
# STEP 3B — Embed the full patent corpus

import numpy as np

texts = chunks_df["text"].tolist()

print(f"Embedding {len(texts):,} chunks...")
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")

embeddings = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)

# Store embeddings in memory
chunks_df["embedding"] = list(embeddings)

print("\nEmbedding complete")
print("-" * 50)
print("Chunks:", len(embeddings))
print("Dimensions:", embeddings.shape[1])
print("Shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)

Embedding 6,964 chunks...
Model: BAAI/bge-base-en-v1.5
Device: mps


Batches:   0%|          | 0/218 [00:00<?, ?it/s]


Embedding complete
--------------------------------------------------
Chunks: 6964
Dimensions: 768
Shape: (6964, 768)
Dtype: float32


In [14]:
# STEP 4 — In-memory semantic retrieval

import numpy as np


def search_patents(query, top_k=5):
    """
    Semantic search directly against the in-memory embeddings.
    No database or vector store required.
    """

    # Embed query
    query_embedding = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    # Cosine similarity because embeddings are normalized
    scores = embeddings @ query_embedding

    # Top results
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = chunks_df.iloc[top_indices].copy()
    results["similarity"] = scores[top_indices]

    return results.reset_index(drop=True)


# ---------------------------------------------------------
# TEST RETRIEVAL
# ---------------------------------------------------------

queries = [
    "advanced semiconductor packaging using through-silicon vias",
    "methods for improving transistor performance and reducing leakage",
    "DRAM memory cell architecture",
    "silicon photonics optical communication",
    "power semiconductor devices for high voltage applications",
]

for query in queries:

    print("\n" + "=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    results = search_patents(query, top_k=3)

    for i, row in results.iterrows():

        print(f"\n[{i + 1}] Similarity: {row['similarity']:.4f}")
        print("Patent:", row["document_id"])
        print("Category:", row["category"])
        print("Section:", row["section"])
        print("Text:", row["text"][:500].replace("\n", " "))


QUERY: advanced semiconductor packaging using through-silicon vias

[1] Similarity: 0.7577
Patent: 6ebfa06f472a9c0d84bc
Category: advanced_packaging
Section: BACKGROUND
Text: [0001] Nanoscale technologies, similar to other technologies, face challenges in high power consumption and signal delay due to global interconnects, but due to their scope, addressing these challenges opens new areas of research. One research area is through silicon via (TSV) based 3D integrated chip (IC) technology, where each layer or stratum is fabricated separately and subsequently vertically integrated. With 3D ICs, the fabrication of disparate strata and the final system integration may b

[2] Similarity: 0.7358
Patent: 7003d3f3c5a05c13a5f5
Category: advanced_packaging
Section: BACKGROUND
Text: 1. Field of Invention   The present invention relates to integrated circuit packaging, and more particularly to multiple die packaging.   2. Related Art   Semiconductor die or chip packages are used to protect the s

In [15]:
# STEP 5 — Improved retrieval with document diversity

def search_patents(query, top_k=5, candidate_k=30):
    """
    Semantic retrieval with document-level diversification.
    No database/vector store required.
    """

    query_embedding = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    # Similarity against all chunks
    scores = embeddings @ query_embedding

    # Get a larger candidate pool first
    candidate_indices = np.argsort(scores)[::-1][:candidate_k]

    results = []

    seen_documents = set()

    for idx in candidate_indices:

        row = chunks_df.iloc[idx]
        document_id = row["document_id"]

        # Avoid returning multiple chunks from the same patent
        if document_id in seen_documents:
            continue

        seen_documents.add(document_id)

        results.append({
            "document_id": document_id,
            "category": row["category"],
            "cpc_section": row["cpc_section"],
            "section": row["section"],
            "similarity": float(scores[idx]),
            "abstract": row["abstract"],
            "text": row["text"],
        })

        if len(results) >= top_k:
            break

    return pd.DataFrame(results)


# ---------------------------------------------------------
# TEST
# ---------------------------------------------------------

query = "methods for reducing leakage and improving transistor performance"

results = search_patents(query, top_k=5)

print("QUERY:")
print(query)

print("\n" + "=" * 100)

for i, row in results.iterrows():

    print(f"\n[{i + 1}] Similarity: {row['similarity']:.4f}")
    print("Patent:", row["document_id"])
    print("Category:", row["category"])
    print("CPC:", row["cpc_section"])
    print("Section:", row["section"])

    print("\nAbstract:")
    print(row["abstract"][:500].replace("\n", " "))

    print("\nRetrieved passage:")
    print(row["text"][:700].replace("\n", " "))

    print("\n" + "-" * 80)

QUERY:
methods for reducing leakage and improving transistor performance


[1] Similarity: 0.7768
Patent: 7a624e78d2ae6904
Category: semiconductor_manufacturing
CPC: 
Section: DESCRIPTION

Abstract:
In a method of fabricating a thin film transistor array substrate, a black matrix having separated portions is first formed on a substrate with an opaque conductive material. An insulation layer is formed to cover the black matrix. A gate line assembly and a data line assembly are formed over the black matrix and insulation layer. Buffer layers are formed to cover the gaps between the separate portions of the black matrix. The buffer conductive layers are formed at the same plane as the gate lin

Retrieved passage:
can be further reduced while completely intercepting the passage of current leakage at the D region. As described above, the black matrix is formed at the TFT array substrate with the pixel electrodes so that opening ratio of the device can be enhanced in a stable manner. The sem

In [16]:
# STEP 6 — Retrieval benchmark

benchmark_queries = [
    "How can through-silicon vias improve advanced semiconductor packaging?",
    "What techniques are used to reduce leakage current in transistor devices?",
    "What structures are used to improve DRAM memory cell performance?",
    "How are semiconductor devices fabricated using multiple material layers?",
    "What approaches improve thermal management in semiconductor packages?",
    "How are silicon photonic devices used for optical communication?",
    "What semiconductor structures are designed for high-voltage power applications?",
    "How can semiconductor devices improve computing performance using advanced architectures?",
    "What techniques improve transistor switching speed?",
    "What methods are used to improve semiconductor manufacturing yield?",
]

benchmark_results = []

for query in benchmark_queries:

    results = search_patents(
        query,
        top_k=5,
        candidate_k=50
    )

    print("\n" + "=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    for rank, (_, row) in enumerate(results.iterrows(), start=1):

        print(
            f"{rank}. "
            f"{row['document_id']} | "
            f"{row['category']} | "
            f"{row['section']} | "
            f"{row['similarity']:.4f}"
        )

        print(
            "   ",
            row["text"][:250]
            .replace("\n", " ")
        )

        benchmark_results.append({
            "query": query,
            "rank": rank,
            "document_id": row["document_id"],
            "category": row["category"],
            "section": row["section"],
            "similarity": row["similarity"],
        })


benchmark_df = pd.DataFrame(benchmark_results)

print("\n\nBenchmark complete")
print("-" * 50)
print("Queries:", len(benchmark_queries))
print("Results/query:", 5)
print("Total retrievals:", len(benchmark_df))


QUERY: How can through-silicon vias improve advanced semiconductor packaging?
1. 6ebfa06f472a9c0d84bc | advanced_packaging | BACKGROUND | 0.7462
    [0001] Nanoscale technologies, similar to other technologies, face challenges in high power consumption and signal delay due to global interconnects, but due to their scope, addressing these challenges opens new areas of research. One research area i
2. a9961143e9743d4b | advanced_packaging | DESCRIPTION | 0.7415
    RELATED APPLICATIONS This application claims priority to U.S. Provisional Patent Application 61/722,738, entitled “METHOD AND STRUCTURE FOR PROVIDING REDUCED CONTACT TESTING AT WAFER SORT USING ELECTRON BEAM DEFLECTION” filed Nov. 5, 2012. FIELD OF T
3. f129a885d93007f261b9 | advanced_packaging | BACKGROUND | 0.7402
    [0001] The present invention relates to a semiconductor structure, and more specifically, to a protected through silicon via (TSV) for providing vertical interconnection in a semiconductor structure.   [0002] 

In [17]:
import subprocess

result = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True
)

print(result.stdout)

NAME                       ID              SIZE      MODIFIED   
gemma3:4b                  a2af6cc3eb7f    3.3 GB    2 days ago    
qwen3:8b                   500a1f067a9f    5.2 GB    2 days ago    
gemma3:1b                  8648f39daa8f    815 MB    3 days ago    
llama3.2:1b                baf6a787fdff    1.3 GB    3 days ago    
qwen2.5:0.5b               a8b0c5157701    397 MB    3 days ago    
nomic-embed-text:latest    0a109f422b47    274 MB    3 days ago    
qwen3.5:0.8b               f3817196d142    1.0 GB    4 days ago    
llama3.1:8b                46e0c10c039e    4.9 GB    6 days ago    



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
# STEP 7 — Connect Ollama to the RAG

import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
LLM_MODEL = "qwen3:8b"


def ask_ollama(prompt, model=LLM_MODEL):
    """Send a prompt to the local Ollama model."""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
        },
        timeout=120,
    )

    response.raise_for_status()

    return response.json()["response"]


# Simple connectivity test
test_response = ask_ollama(
    "In one sentence, explain what a semiconductor is."
)

print("Ollama connection successful")
print("Model:", LLM_MODEL)
print("Response:", test_response)

Ollama connection successful
Model: qwen3:8b
Response: A semiconductor is a material with electrical conductivity between that of conductors and insulators, enabling it to control the flow of electric current in electronic devices.


In [19]:
def answer_with_context(question: str, context: str) -> str:
    """Generate an answer using retrieved RAG context."""

    prompt = f"""
You are a semiconductor patent research assistant.

Answer the user's question using ONLY the information provided
in the retrieved patent context below.

If the context does not contain enough information to answer,
say so explicitly.

Retrieved patent context:
-------------------------
{context}
-------------------------

User question:
{question}

Provide a concise, technically accurate answer.
"""

    return ask_ollama(prompt)

In [20]:
context = """
US Patent XXXXX describes a semiconductor manufacturing process
using a plasma etching step followed by deposition of a dielectric
layer...
"""

question = "What manufacturing process is described in this patent?"

answer = answer_with_context(question, context)

print(answer)

The patent describes a semiconductor manufacturing process involving a plasma etching step followed by the deposition of a dielectric layer.


In [23]:
# STEP 8 — RAG Prompt Component

def build_rag_prompt(question, retrieved_context):
    """
    Build a grounded prompt for the local Ollama model.
    """

    prompt = f"""
You are a semiconductor patent research assistant.

Use ONLY the retrieved patent information provided below
to answer the question.

If the retrieved information does not contain enough evidence,
say that the available patent context is insufficient.

Do not invent patent numbers, claims, inventors, technical details,
or conclusions that are not supported by the retrieved context.

RETRIEVED PATENT CONTEXT
========================

{retrieved_context}

========================

QUESTION
========

{question}

ANSWER
======

Provide a clear, technically accurate answer based only on
the retrieved patent context.
"""

    return prompt

In [25]:
# Create test context

test_context = """
Patent ID: US1234567B2

Title:
Semiconductor manufacturing process

Abstract:
The patent describes a semiconductor fabrication process
using plasma etching followed by deposition of a dielectric layer.

Claim 1:
A method comprising etching a semiconductor substrate using
plasma and subsequently depositing a dielectric material.
"""

test_question = "What manufacturing process is described?"

prompt = build_rag_prompt(
    test_question,
    test_context
)

print(prompt)


You are a semiconductor patent research assistant.

Use ONLY the retrieved patent information provided below
to answer the question.

If the retrieved information does not contain enough evidence,
say that the available patent context is insufficient.

Do not invent patent numbers, claims, inventors, technical details,
or conclusions that are not supported by the retrieved context.

RETRIEVED PATENT CONTEXT


Patent ID: US1234567B2

Title:
Semiconductor manufacturing process

Abstract:
The patent describes a semiconductor fabrication process
using plasma etching followed by deposition of a dielectric layer.

Claim 1:
A method comprising etching a semiconductor substrate using
plasma and subsequently depositing a dielectric material.



QUESTION

What manufacturing process is described?

ANSWER

Provide a clear, technically accurate answer based only on
the retrieved patent context.



In [26]:
# STEP 9 — Generate Answer from RAG Context

answer = ask_ollama(prompt)

print("RAG Answer:")
print(answer)

RAG Answer:
The manufacturing process described is a semiconductor fabrication method involving **plasma etching** of a semiconductor substrate followed by the **deposition of a dielectric material**. This is explicitly outlined in the patent's claims and abstract.


In [30]:
# STEP 10 — Inspect Existing RAG Components

print("Available RAG-related objects:\n")

for name, obj in sorted(globals().items()):
    if any(term in name.lower() for term in [
        "embed",
        "vector",
        "index",
        "chunk",
        "document",
        "retriev",
        "faiss",
        "chroma",
        "collection",
        "metadata",
        "patent",
    ]):
        print(f"{name:<30} {type(obj)}")

Available RAG-related objects:

CHUNK_OVERLAP                  <class 'int'>
CHUNK_SIZE                     <class 'int'>
chunk                          <class 'str'>
chunk_index                    <class 'int'>
chunk_text                     <class 'function'>
chunks                         <class 'list'>
chunks_df                      <class 'pandas.core.frame.DataFrame'>
document_id                    <class 'str'>
documents_with_markers         <class 'int'>
embeddings                     <class 'numpy.ndarray'>
search_patents                 <class 'function'>
section_chunks                 <class 'list'>
test_embeddings                <class 'numpy.ndarray'>


In [31]:
# Inspect imported libraries that may be relevant

print("\nRelevant modules:")

for name, obj in sorted(globals().items()):
    if name in ["faiss", "chromadb", "numpy", "np", "pandas", "pd"]:
        print(f"{name:<15} {type(obj)}")


Relevant modules:
np              <class 'module'>
pd              <class 'module'>


In [33]:
# ============================================================
# STEP 10 — Load Semiconductor Patent Corpus
# ============================================================

from pathlib import Path
import pandas as pd

# Your existing semiconductor patent corpus
CORPUS_ROOT = Path(
    "/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents"
)

METADATA_FILE = CORPUS_ROOT / "metadata.csv"

print("Corpus exists:", CORPUS_ROOT.exists())
print("Metadata exists:", METADATA_FILE.exists())

if not CORPUS_ROOT.exists():
    raise FileNotFoundError(
        f"Patent corpus not found: {CORPUS_ROOT}"
    )

if not METADATA_FILE.exists():
    raise FileNotFoundError(
        f"Metadata file not found: {METADATA_FILE}"
    )

# Load metadata
metadata = pd.read_csv(METADATA_FILE)

print("\nPatent metadata loaded successfully.")
print("Number of records:", len(metadata))

print("\nColumns:")
for column in metadata.columns:
    print(" -", column)

print("\nFirst 5 records:")
display(metadata.head())

Corpus exists: True
Metadata exists: True

Patent metadata loaded successfully.
Number of records: 500

Columns:
 - document_id
 - publication_number
 - title
 - abstract
 - claims
 - description
 - cpc_section
 - inventors
 - assignee
 - filing_date
 - publication_date
 - category
 - filename
 - source
 - word_count
 - character_count

First 5 records:


,document_id,publication_number,title,abstract,claims,description,cpc_section,inventors,assignee,filing_date,publication_date,category,filename,source,word_count,character_count
0,035ed7683a8cf2c3,NaN,NaN,A method for semiconductor device feature deve...,NaN,FIELD OF THE INVENTION [0001] This invention g...,NaN,NaN,NaN,NaN,NaN,semiconductor_manufacturing,patent_035ed7683a8cf2c3.txt,BIGPATENT,3538,22987
1,08064dc5e6a41c78,NaN,NaN,A pattern generator for integrated multilayer ...,NaN,BACKGROUND OF THE INVENTION 1. Field of the In...,NaN,NaN,NaN,NaN,NaN,semiconductor_manufacturing,patent_08064dc5e6a41c78.txt,BIGPATENT,6450,38272
2,095d694dbb617b65,NaN,NaN,The present invention has an object to provide...,NaN,BACKGROUND OF THE INVENTION [0001] 1. Field of...,NaN,NaN,NaN,NaN,NaN,semiconductor_manufacturing,patent_095d694dbb617b65.txt,BIGPATENT,12363,73608
3,0bf59cde519a6316,NaN,NaN,A technique for measuring the ion implant dosa...,NaN,BACKGROUND OF THE INVENTION 1. Field of the In...,NaN,NaN,NaN,NaN,NaN,semiconductor_manufacturing,patent_0bf59cde519a6316.txt,BIGPATENT,1957,11786
4,0c3c5540a4fd4e21,NaN,NaN,A method of and apparatus for measuring line p...,NaN,CROSS-REFERENCE TO RELATED APPLICATIONS This a...,NaN,NaN,NaN,NaN,NaN,semiconductor_manufacturing,patent_0c3c5540a4fd4e21.txt,BIGPATENT,6365,40359


In [34]:
# ============================================================
# STEP 10B — Inspect Patent Files
# ============================================================

all_files = [
    path for path in CORPUS_ROOT.rglob("*")
    if path.is_file()
]

print("Total files:", len(all_files))

print("\nFile types:")
file_types = {}

for path in all_files:
    suffix = path.suffix.lower() or "[no extension]"
    file_types[suffix] = file_types.get(suffix, 0) + 1

for suffix, count in sorted(file_types.items()):
    print(f"{suffix:15} {count}")

print("\nExample files:")
for path in all_files[:20]:
    print(path.relative_to(CORPUS_ROOT))

Total files: 501

File types:
.csv            1
.txt            500

Example files:
metadata.csv
advanced_packaging/patent_23ebeda5c273212c3bcd.txt
advanced_packaging/patent_d6b919ccb69b6b48ba28.txt
advanced_packaging/patent_77ef8db797ce48c60943.txt
advanced_packaging/patent_8f954a54fe074893c736.txt
advanced_packaging/patent_403a180868f181ccb1a0.txt
advanced_packaging/patent_da2d58c881907dcfe7fc.txt
advanced_packaging/patent_24a95eaf36a613e9a7ec.txt
advanced_packaging/patent_b1aaa836bc3709df1d70.txt
advanced_packaging/patent_6412ebaec693a0105f28.txt
advanced_packaging/patent_3d5e0281a6e4e66c129d.txt
advanced_packaging/patent_51de540a07c2efbb0473.txt
advanced_packaging/patent_c182265c4cd46ad5.txt
advanced_packaging/patent_5469465e8f806fc2b1fe.txt
advanced_packaging/patent_6001024cc3df05a76add.txt
advanced_packaging/patent_6c78e01a8516125d.txt
advanced_packaging/patent_a969f5fbe709d62ce477.txt
advanced_packaging/patent_de0145519ce9aacf.txt
advanced_packaging/patent_313810c0b6e6e16b1b9b.t

In [35]:
# ============================================================
# STEP 11 — Load Patent Documents
# ============================================================

from pathlib import Path
import pandas as pd

# Reuse variables if they already exist
CORPUS_ROOT = Path(
    "/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents"
)

METADATA_FILE = CORPUS_ROOT / "metadata.csv"

metadata = pd.read_csv(METADATA_FILE)

print("Metadata records:", len(metadata))
print("Columns:", metadata.columns.tolist())

Metadata records: 500
Columns: ['document_id', 'publication_number', 'title', 'abstract', 'claims', 'description', 'cpc_section', 'inventors', 'assignee', 'filing_date', 'publication_date', 'category', 'filename', 'source', 'word_count', 'character_count']


In [36]:
# Show a representative record

display(
    metadata.head(3).T
)

,0,1,2
document_id,035ed7683a8cf2c3,08064dc5e6a41c78,095d694dbb617b65
publication_number,NaN,NaN,NaN
title,NaN,NaN,NaN
abstract,A method for semiconductor device feature deve...,A pattern generator for integrated multilayer ...,The present invention has an object to provide...
claims,NaN,NaN,NaN
description,FIELD OF THE INVENTION [0001] This invention g...,BACKGROUND OF THE INVENTION 1. Field of the In...,BACKGROUND OF THE INVENTION [0001] 1. Field of...
cpc_section,NaN,NaN,NaN
inventors,NaN,NaN,NaN
assignee,NaN,NaN,NaN
filing_date,NaN,NaN,NaN


In [37]:
# ============================================================
# STEP 11B — Find Patent Text Files
# ============================================================

text_extensions = {
    ".txt",
    ".text",
    ".md",
    ".xml",
    ".json",
}

patent_files = [
    path
    for path in CORPUS_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in text_extensions
    and path.name != "metadata.csv"
]

print("Patent text files found:", len(patent_files))

print("\nExamples:")
for path in patent_files[:10]:
    print(path.relative_to(CORPUS_ROOT))

Patent text files found: 500

Examples:
advanced_packaging/patent_23ebeda5c273212c3bcd.txt
advanced_packaging/patent_d6b919ccb69b6b48ba28.txt
advanced_packaging/patent_77ef8db797ce48c60943.txt
advanced_packaging/patent_8f954a54fe074893c736.txt
advanced_packaging/patent_403a180868f181ccb1a0.txt
advanced_packaging/patent_da2d58c881907dcfe7fc.txt
advanced_packaging/patent_24a95eaf36a613e9a7ec.txt
advanced_packaging/patent_b1aaa836bc3709df1d70.txt
advanced_packaging/patent_6412ebaec693a0105f28.txt
advanced_packaging/patent_3d5e0281a6e4e66c129d.txt


In [38]:
# ============================================================
# STEP 11C — Inspect One Patent
# ============================================================

if not patent_files:
    raise FileNotFoundError(
        "No patent text files were found."
    )

sample_file = patent_files[0]

print("Sample file:")
print(sample_file)

print("\nFile contents:\n")
print(sample_file.read_text(
    encoding="utf-8",
    errors="ignore"
)[:5000])

Sample file:
/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents/advanced_packaging/patent_23ebeda5c273212c3bcd.txt

File contents:

PATENT DOCUMENT

DOCUMENT ID: 23ebeda5c273212c3bcd
CPC SECTION: H
CATEGORY: advanced_packaging
SOURCE: BIGPATENT

ABSTRACT

A power interconnection system comprising a plurality of z-axis compliant connectors passing power and ground signals between a first circuit board to a second circuit board is disclosed. The interconnection system provides for an extremely low impedance through a broad range of frequencies and allows for large amounts of current to pass from one substrate to the next either statically or dynamically. The interconnection system may be located close to the die or may be further away depending upon the system requirements. The interconnection may also be used to take up mechanical tolerances between the two substrates while providing a low impedance. interconnect.

DETAILED DESCRIPTION

CROSS-REFERENCE TO RELATED APPL

In [39]:
# ============================================================
# STEP 12 — Parse and Chunk Patent Documents
# ============================================================

from pathlib import Path
import re
import pandas as pd


def parse_patent_file(file_path):
    """
    Parse a semiconductor patent text file and preserve
    its metadata and major sections.
    """

    text = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    # --------------------------------------------------------
    # Extract document metadata
    # --------------------------------------------------------

    document_id = re.search(
        r"DOCUMENT ID:\s*(.+)",
        text
    )

    cpc_section = re.search(
        r"CPC SECTION:\s*(.+)",
        text
    )

    category = re.search(
        r"CATEGORY:\s*(.+)",
        text
    )

    source = re.search(
        r"SOURCE:\s*(.+)",
        text
    )

    document_id = (
        document_id.group(1).strip()
        if document_id else file_path.stem
    )

    cpc_section = (
        cpc_section.group(1).strip()
        if cpc_section else None
    )

    category = (
        category.group(1).strip()
        if category else None
    )

    source = (
        source.group(1).strip()
        if source else None
    )

    # --------------------------------------------------------
    # Extract abstract
    # --------------------------------------------------------

    abstract_match = re.search(
        r"ABSTRACT\s*=+\s*(.*?)(?=\n[A-Z][A-Z\s]+={3,}|\Z)",
        text,
        flags=re.DOTALL
    )

    abstract = (
        abstract_match.group(1).strip()
        if abstract_match else ""
    )

    # --------------------------------------------------------
    # Extract detailed description
    # --------------------------------------------------------

    description_match = re.search(
        r"DETAILED DESCRIPTION\s*=+\s*(.*)",
        text,
        flags=re.DOTALL
    )

    description = (
        description_match.group(1).strip()
        if description_match else ""
    )

    # --------------------------------------------------------
    # Extract claims if present
    # --------------------------------------------------------

    claims_match = re.search(
        r"CLAIMS?\s*=+\s*(.*)",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    claims = (
        claims_match.group(1).strip()
        if claims_match else ""
    )

    return {
        "document_id": document_id,
        "cpc_section": cpc_section,
        "category": category,
        "source": source,
        "abstract": abstract,
        "description": description,
        "claims": claims,
        "file_path": str(file_path),
    }

In [40]:
# ============================================================
# STEP 12B — Test Patent Parser
# ============================================================

parsed_patent = parse_patent_file(sample_file)

for key, value in parsed_patent.items():
    if key not in {"description", "claims"}:
        print(f"{key}: {value}")

print("\nDescription characters:", len(parsed_patent["description"]))
print("Claims characters:", len(parsed_patent["claims"]))

document_id: 23ebeda5c273212c3bcd
cpc_section: H
category: advanced_packaging
source: BIGPATENT
abstract: A power interconnection system comprising a plurality of z-axis compliant connectors passing power and ground signals between a first circuit board to a second circuit board is disclosed. The interconnection system provides for an extremely low impedance through a broad range of frequencies and allows for large amounts of current to pass from one substrate to the next either statically or dynamically. The interconnection system may be located close to the die or may be further away depending upon the system requirements. The interconnection may also be used to take up mechanical tolerances between the two substrates while providing a low impedance. interconnect.
file_path: /Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents/advanced_packaging/patent_23ebeda5c273212c3bcd.txt

Description characters: 78512
Claims characters: 0


In [41]:
# ============================================================
# STEP 12C — Create Patent Chunks
# ============================================================

def chunk_text(text, chunk_size=1500, overlap=250):
    """
    Split text into overlapping chunks.

    chunk_size:
        Approximate number of characters per chunk.

    overlap:
        Number of characters shared between adjacent chunks.
    """

    if not text:
        return []

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

In [42]:
# ============================================================
# STEP 12D — Build Structured Patent Chunks
# ============================================================

def create_patent_chunks(parsed_patent, chunk_size=1500, overlap=250):
    """
    Convert a parsed patent into RAG-ready chunks while
    preserving patent metadata.
    """

    chunks = []

    sections = {
        "abstract": parsed_patent["abstract"],
        "description": parsed_patent["description"],
        "claims": parsed_patent["claims"],
    }

    chunk_id = 0

    for section_name, section_text in sections.items():

        section_chunks = chunk_text(
            section_text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk in section_chunks:

            chunks.append({
                "chunk_id": chunk_id,
                "document_id": parsed_patent["document_id"],
                "category": parsed_patent["category"],
                "cpc_section": parsed_patent["cpc_section"],
                "source": parsed_patent["source"],
                "section": section_name,
                "text": chunk,
                "file_path": parsed_patent["file_path"],
            })

            chunk_id += 1

    return chunks

In [43]:
# ============================================================
# STEP 12E — Test Chunking
# ============================================================

patent_chunks = create_patent_chunks(parsed_patent)

print("Document:", parsed_patent["document_id"])
print("Number of chunks:", len(patent_chunks))

print("\nFirst chunk:")
print(patent_chunks[0])

Document: 23ebeda5c273212c3bcd
Number of chunks: 64

First chunk:
{'chunk_id': 0, 'document_id': '23ebeda5c273212c3bcd', 'category': 'advanced_packaging', 'cpc_section': 'H', 'source': 'BIGPATENT', 'section': 'abstract', 'text': 'A power interconnection system comprising a plurality of z-axis compliant connectors passing power and ground signals between a first circuit board to a second circuit board is disclosed. The interconnection system provides for an extremely low impedance through a broad range of frequencies and allows for large amounts of current to pass from one substrate to the next either statically or dynamically. The interconnection system may be located close to the die or may be further away depending upon the system requirements. The interconnection may also be used to take up mechanical tolerances between the two substrates while providing a low impedance. interconnect.', 'file_path': '/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents/advanced_pack

In [44]:
# ============================================================
# STEP 13 — Process Entire Semiconductor Patent Corpus
# ============================================================

from pathlib import Path
import pandas as pd

# Find all patent text files
patent_files = [
    path
    for path in CORPUS_ROOT.rglob("*.txt")
    if path.name != "metadata.csv"
]

print("Patent files found:", len(patent_files))

# ------------------------------------------------------------
# Parse and chunk every patent
# ------------------------------------------------------------

all_chunks = []

failed_files = []

for i, file_path in enumerate(patent_files, start=1):

    try:
        parsed = parse_patent_file(file_path)

        chunks = create_patent_chunks(
            parsed,
            chunk_size=1500,
            overlap=250,
        )

        all_chunks.extend(chunks)

    except Exception as e:
        failed_files.append({
            "file": str(file_path),
            "error": str(e),
        })

    if i % 50 == 0 or i == len(patent_files):
        print(
            f"Processed {i}/{len(patent_files)} patents | "
            f"Chunks: {len(all_chunks)}"
        )


print("\n========================================")
print("PROCESSING COMPLETE")
print("========================================")

print("Patents processed:", len(patent_files))
print("Total chunks:", len(all_chunks))
print("Failed files:", len(failed_files))

Patent files found: 500
Processed 50/500 patents | Chunks: 1103
Processed 100/500 patents | Chunks: 1605
Processed 150/500 patents | Chunks: 1656
Processed 200/500 patents | Chunks: 1708
Processed 250/500 patents | Chunks: 1758
Processed 300/500 patents | Chunks: 1809
Processed 350/500 patents | Chunks: 1860
Processed 400/500 patents | Chunks: 1910
Processed 450/500 patents | Chunks: 1960
Processed 500/500 patents | Chunks: 2093

PROCESSING COMPLETE
Patents processed: 500
Total chunks: 2093
Failed files: 0


In [45]:
# ============================================================
# STEP 13B — Create RAG Chunk DataFrame
# ============================================================

chunks_df = pd.DataFrame(all_chunks)

print("Shape:", chunks_df.shape)

print("\nColumns:")
print(chunks_df.columns.tolist())

display(chunks_df.head())

Shape: (2093, 8)

Columns:
['chunk_id', 'document_id', 'category', 'cpc_section', 'source', 'section', 'text', 'file_path']


,chunk_id,document_id,category,cpc_section,source,section,text,file_path
0,0,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,abstract,A power interconnection system comprising a pl...,/Users/jamesjr/Documents/SemiconductorPatentRA...
1,1,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description,CROSS-REFERENCE TO RELATED APPLICATIONS \n [00...,/Users/jamesjr/Documents/SemiconductorPatentRA...
2,2,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description,"RMAL AND EMI MANAGEMENT,” filed Feb. 16, 2001,...",/Users/jamesjr/Documents/SemiconductorPatentRA...
3,3,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description,"ation No. 60/219,813, entitled “HIGH-CURRENT M...",/Users/jamesjr/Documents/SemiconductorPatentRA...
4,4,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description,"ARRAY,” filed Aug. 2, 2001, now U.S. Pat. No. ...",/Users/jamesjr/Documents/SemiconductorPatentRA...


In [46]:
# ============================================================
# STEP 13C — Inspect Chunk Distribution
# ============================================================

print("Chunks by section:")
display(
    chunks_df["section"]
    .value_counts()
    .to_frame("count")
)

print("\nChunks by category:")
display(
    chunks_df["category"]
    .value_counts()
    .to_frame("count")
)

Chunks by section:


,count
section,
description,1588
abstract,505



Chunks by category:


,count
category,
advanced_packaging,1580
power_semiconductors,133
transistor_device_technology,101
semiconductor_manufacturing,101
memory,76
ai_advanced_semiconductor,52
photonics,50


In [47]:
# ============================================================
# STEP 13D — Verify Patent Metadata
# ============================================================

print("Unique patents:", chunks_df["document_id"].nunique())

print("\nExample patent IDs:")
display(
    chunks_df[
        [
            "document_id",
            "category",
            "cpc_section",
            "source",
            "section",
        ]
    ].head(10)
)

Unique patents: 500

Example patent IDs:


,document_id,category,cpc_section,source,section
0,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,abstract
1,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description
2,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description
3,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description
4,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description
5,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description
6,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description
7,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description
8,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description
9,23ebeda5c273212c3bcd,advanced_packaging,H,BIGPATENT,description


In [48]:
# ============================================================
# STEP 13E — Data Quality Check
# ============================================================

empty_text = chunks_df["text"].isna() | (
    chunks_df["text"].str.strip() == ""
)

print("Empty chunks:", empty_text.sum())

print(
    "Average chunk length:",
    chunks_df["text"].str.len().mean()
)

print(
    "Minimum chunk length:",
    chunks_df["text"].str.len().min()
)

print(
    "Maximum chunk length:",
    chunks_df["text"].str.len().max()
)

Empty chunks: 0
Average chunk length: 1287.583850931677
Minimum chunk length: 193
Maximum chunk length: 1500


In [49]:
# ============================================================
# STEP 14 — Connect Ollama Embeddings
# ============================================================

import requests
import numpy as np

OLLAMA_EMBED_URL = "http://localhost:11434/api/embeddings"
EMBED_MODEL = "nomic-embed-text"


def embed_text(text, model=EMBED_MODEL):
    """Create a local embedding using Ollama."""

    response = requests.post(
        OLLAMA_EMBED_URL,
        json={
            "model": model,
            "prompt": text,
        },
        timeout=120,
    )

    response.raise_for_status()

    return response.json()["embedding"]

In [50]:
# ============================================================
# STEP 14B — Test Embedding
# ============================================================

test_text = chunks_df.iloc[0]["text"]

test_embedding = embed_text(test_text)

print("Embedding model:", EMBED_MODEL)
print("Embedding dimensions:", len(test_embedding))
print("First 10 values:")
print(test_embedding[:10])

Embedding model: nomic-embed-text
Embedding dimensions: 768
First 10 values:
[0.9271228313446045, 0.7600730657577515, -3.2690510749816895, -1.303018569946289, 0.4443703889846802, -0.3875020742416382, 0.7055721282958984, 0.06263209134340286, -0.8916878700256348, -0.9662848711013794]


In [51]:
# ============================================================
# STEP 14C — Embed All Patent Chunks
# ============================================================

embeddings = []

total = len(chunks_df)

for i, text in enumerate(chunks_df["text"], start=1):

    embedding = embed_text(text)

    embeddings.append(embedding)

    if i % 50 == 0 or i == total:
        print(f"Embedded {i}/{total} chunks")

Embedded 50/2093 chunks
Embedded 100/2093 chunks
Embedded 150/2093 chunks
Embedded 200/2093 chunks
Embedded 250/2093 chunks
Embedded 300/2093 chunks
Embedded 350/2093 chunks
Embedded 400/2093 chunks
Embedded 450/2093 chunks
Embedded 500/2093 chunks
Embedded 550/2093 chunks
Embedded 600/2093 chunks
Embedded 650/2093 chunks
Embedded 700/2093 chunks
Embedded 750/2093 chunks
Embedded 800/2093 chunks
Embedded 850/2093 chunks
Embedded 900/2093 chunks
Embedded 950/2093 chunks
Embedded 1000/2093 chunks
Embedded 1050/2093 chunks
Embedded 1100/2093 chunks
Embedded 1150/2093 chunks
Embedded 1200/2093 chunks
Embedded 1250/2093 chunks
Embedded 1300/2093 chunks
Embedded 1350/2093 chunks
Embedded 1400/2093 chunks
Embedded 1450/2093 chunks
Embedded 1500/2093 chunks
Embedded 1550/2093 chunks
Embedded 1600/2093 chunks
Embedded 1650/2093 chunks
Embedded 1700/2093 chunks
Embedded 1750/2093 chunks
Embedded 1800/2093 chunks
Embedded 1850/2093 chunks
Embedded 1900/2093 chunks
Embedded 1950/2093 chunks
Embedd

In [52]:
# ============================================================
# STEP 14D — Create Embedding Matrix
# ============================================================

embedding_matrix = np.array(
    embeddings,
    dtype=np.float32
)

print("Embedding matrix shape:")
print(embedding_matrix.shape)

print("Data type:")
print(embedding_matrix.dtype)

Embedding matrix shape:
(2093, 768)
Data type:
float32


In [53]:
# ============================================================
# STEP 14E — Normalize Embeddings
# ============================================================

norms = np.linalg.norm(
    embedding_matrix,
    axis=1,
    keepdims=True
)

embedding_matrix_normalized = (
    embedding_matrix / np.maximum(norms, 1e-12)
)

print(
    "Normalized embedding matrix:",
    embedding_matrix_normalized.shape
)

Normalized embedding matrix: (2093, 768)


In [54]:
# ============================================================
# STEP 15 — Semantic Patent Retrieval
# ============================================================

def retrieve_patents(query, top_k=5):
    """
    Retrieve the most semantically similar patent chunks
    for a natural-language query.
    """

    # Embed the query using the same embedding model
    query_embedding = np.array(
        embed_text(query),
        dtype=np.float32
    )

    # Normalize query embedding
    query_norm = np.linalg.norm(query_embedding)

    query_embedding = (
        query_embedding / max(query_norm, 1e-12)
    )

    # Cosine similarity because both vectors are normalized
    similarities = embedding_matrix_normalized @ query_embedding

    # Get indices of highest similarity
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Build structured results
    results = chunks_df.iloc[top_indices].copy()

    results["similarity"] = similarities[top_indices]

    # Reset index for cleaner inspection
    results = results.reset_index(drop=True)

    return results

In [57]:
# ============================================================
# STEP 15C — Display Retrieved Patents
# ============================================================

def display_retrieval_results(results):
    """
    Display retrieved patent chunks safely.

    Only displays metadata columns that actually exist.
    """

    for i, row in results.iterrows():

        print("=" * 80)
        print(f"RESULT {i + 1}")
        print("=" * 80)

        # Core metadata
        print("Patent ID :", row.get("document_id", "N/A"))
        print("Category  :", row.get("category", "N/A"))
        print("Section   :", row.get("section", "N/A"))

        # Optional metadata
        if "source" in results.columns:
            print("Source    :", row["source"])

        if "cpc_section" in results.columns:
            print("CPC       :", row["cpc_section"])

        if "similarity" in results.columns:
            print(
                "Similarity:",
                round(float(row["similarity"]), 4)
            )

        print("\nText:")
        print(row.get("text", ""))

        print()

In [58]:
display_retrieval_results(results)

RESULT 1
Patent ID : d3ead82de489632b
Category  : semiconductor_manufacturing
Section   : DESCRIPTION
CPC       : 
Similarity: 0.7792

Text:
CROSS-REFERENCE: This continuation patent application claims priority under 35 U.S.C. §120 from U.S. patent application No. 11/166,960, filed Jun. 23, 2005, now issued as U.S. Pat. No.7,685,481 B2, the disclosure of which is incorporated by reference herein in its entirety. BACKGROUND OF THE INVENTION 1. Field of the Invention The present invention relates generally to a system and method for quality control of integrated circuits manufactured by a semiconductor manufacturing process and, more particularly, to a system and method for analyzing defects in integrated circuits manufactured using a semiconductor fabrication process. 2. Description of the Prior Art The semiconductor manufacturing industry is continually evolving its fabrication processes and developing new processes to produce smaller and smaller geometries of the semiconductor devices

In [59]:
# STEP 15D — Inspect Actual Retrieval Schema

print("Result columns:")
print(results.columns.tolist())

print("\nChunk DataFrame columns:")
print(chunks_df.columns.tolist())

print("\nExample retrieved record:")
display(results.head(1).T)

Result columns:
['document_id', 'category', 'cpc_section', 'section', 'similarity', 'abstract', 'text']

Chunk DataFrame columns:
['chunk_id', 'document_id', 'category', 'cpc_section', 'source', 'section', 'text', 'file_path']

Example retrieved record:


,0
document_id,d3ead82de489632b
category,semiconductor_manufacturing
cpc_section,
section,DESCRIPTION
similarity,0.779211
abstract,A system and method for defect analysis are di...
text,CROSS-REFERENCE: This continuation patent appl...


In [60]:
# ============================================================
# STEP 15E — Finalize Patent Retrieval Results
# ============================================================

def retrieve_patents(query, top_k=5):
    """
    Retrieve the most semantically similar patent chunks.

    Returns the original patent metadata together with
    the similarity score.
    """

    # Embed query
    query_embedding = np.array(
        embed_text(query),
        dtype=np.float32
    )

    # Normalize query
    query_norm = np.linalg.norm(query_embedding)

    query_embedding = (
        query_embedding / max(query_norm, 1e-12)
    )

    # Calculate cosine similarity
    similarities = (
        embedding_matrix_normalized @ query_embedding
    )

    # Top-k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Retrieve original records
    results = chunks_df.iloc[top_indices].copy()

    # Add similarity
    results["similarity"] = similarities[top_indices]

    # Reset index
    results = results.reset_index(drop=True)

    return results

In [61]:
# ============================================================
# STEP 15F — Verify Retrieval Schema
# ============================================================

question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

results = retrieve_patents(
    question,
    top_k=5
)

print("Retrieved:", len(results))
print("\nColumns:")
print(results.columns.tolist())

display(
    results[
        [
            "document_id",
            "category",
            "section",
            "similarity",
            "text",
        ]
    ]
)

Retrieved: 5

Columns:
['chunk_id', 'document_id', 'category', 'cpc_section', 'source', 'section', 'text', 'file_path', 'similarity']


,document_id,category,section,similarity,text
0,01ea3f6e9244856597cb,advanced_packaging,description,0.739352,TECHNICAL FIELD \n The invention described her...
1,f2ada019c57f2835065e,advanced_packaging,description,0.732492,"technology that, in the future, may particular..."
2,a969f5fbe709d62ce477,advanced_packaging,description,0.727849,BACKGROUND \n [0001] The present invention rel...
3,01ea3f6e9244856597cb,advanced_packaging,description,0.727297,ine without damage the packaged is “qualified”...
4,b8ead524636a26d39526,advanced_packaging,description,0.726091,wherein the wafer is severed into individual s...


In [62]:
# ============================================================
# STEP 16 — Build Context from Retrieved Patents
# ============================================================

def build_retrieved_context(results):
    """
    Convert retrieved patent chunks into structured
    context for the LLM.
    """

    context_parts = []

    for i, row in results.iterrows():

        context_parts.append(
            f"""
PATENT RESULT {i + 1}

Patent ID: {row["document_id"]}
Category: {row["category"]}
CPC Section: {row["cpc_section"]}
Section: {row["section"]}
Similarity: {row["similarity"]:.4f}

PATENT TEXT:
{row["text"]}
"""
        )

    return "\n".join(context_parts)

In [63]:
# ============================================================
# STEP 16B — Create Real RAG Context
# ============================================================

context = build_retrieved_context(results)

print(context)


PATENT RESULT 1

Patent ID: 01ea3f6e9244856597cb
Category: advanced_packaging
CPC Section: H
Section: description
Similarity: 0.7394

PATENT TEXT:
TECHNICAL FIELD 
 The invention described herein relates generally to semiconductor device packaging. In particular, the invention relates to a method and package for a semiconductor device that is resistant to electrostatic discharge that can be caused by various factors. 
 BACKGROUND 
 The semiconductor industry makes wide use of standard BGA (ball grid array) type semiconductor packages. Such packages generally include a BT (bismaleimide triazine) core having various metallization and solder mask layers to form the substrate. A semiconductor die is attached to the substrate and electrically connected to various electrical connections of the substrate using ball attach or wire-bonding techniques. The wire bonds and the die are typically encapsulated with a protective layer of encapsulant. Such packages and the methods of their constructio

In [64]:
# ============================================================
# STEP 16C — Real RAG Query
# ============================================================

question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

results = retrieve_patents(
    question,
    top_k=5
)

context = build_retrieved_context(results)

prompt = build_rag_prompt(
    question,
    context
)

answer = ask_ollama(prompt)

print("=" * 80)
print("RAG ANSWER")
print("=" * 80)
print(answer)

RAG ANSWER
The available patent context is insufficient to determine techniques used to reduce impedance in semiconductor packaging. The retrieved patents primarily focus on enhancing electrostatic discharge (ESD) resistance, improving high-speed signal transmission, and optimizing packaging structures (e.g., BGA/CSP configurations, encapsulation methods). None of the texts explicitly mention techniques for reducing impedance, such as specific material choices, design modifications, or circuit configurations aimed at lowering electrical resistance. The emphasis is on increasing impedance for ESD protection or managing signal integrity in high-speed applications, not on impedance reduction.


In [65]:
# ============================================================
# STEP 16D — Show Sources
# ============================================================

print("\n")
print("=" * 80)
print("PATENTS USED")
print("=" * 80)

for i, row in results.iterrows():

    print(
        f"{i + 1}. "
        f"{row['document_id']} | "
        f"{row['category']} | "
        f"{row['section']} | "
        f"similarity={row['similarity']:.4f}"
    )



PATENTS USED
1. 01ea3f6e9244856597cb | advanced_packaging | description | similarity=0.7394
2. f2ada019c57f2835065e | advanced_packaging | description | similarity=0.7325
3. a969f5fbe709d62ce477 | advanced_packaging | description | similarity=0.7278
4. 01ea3f6e9244856597cb | advanced_packaging | description | similarity=0.7273
5. b8ead524636a26d39526 | advanced_packaging | description | similarity=0.7261


In [66]:
# STEP 17 — Deduplicated Semantic Retrieval

def retrieve_patents(query, top_k=5):
    """
    Retrieve semantically similar patent chunks while ensuring
    that each patent appears only once.

    The highest-similarity chunk is retained for each patent.
    """

    query_embedding = np.array(
        embed_text(query),
        dtype=np.float32
    )

    query_norm = np.linalg.norm(query_embedding)

    query_embedding = (
        query_embedding / max(query_norm, 1e-12)
    )

    similarities = (
        embedding_matrix_normalized @ query_embedding
    )

    # Rank all chunks by similarity
    ranked_indices = np.argsort(similarities)[::-1]

    selected_indices = []
    seen_patents = set()

    for idx in ranked_indices:

        patent_id = chunks_df.iloc[idx]["document_id"]

        if patent_id in seen_patents:
            continue

        seen_patents.add(patent_id)
        selected_indices.append(idx)

        if len(selected_indices) == top_k:
            break

    results = chunks_df.iloc[selected_indices].copy()

    results["similarity"] = similarities[selected_indices]

    results = results.reset_index(drop=True)

    return results

In [67]:
question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

results = retrieve_patents(
    question,
    top_k=5
)

display_retrieval_results(results)

RESULT 1
Patent ID : 01ea3f6e9244856597cb
Category  : advanced_packaging
Section   : description
Source    : BIGPATENT
CPC       : H
Similarity: 0.7394

Text:
TECHNICAL FIELD 
 The invention described herein relates generally to semiconductor device packaging. In particular, the invention relates to a method and package for a semiconductor device that is resistant to electrostatic discharge that can be caused by various factors. 
 BACKGROUND 
 The semiconductor industry makes wide use of standard BGA (ball grid array) type semiconductor packages. Such packages generally include a BT (bismaleimide triazine) core having various metallization and solder mask layers to form the substrate. A semiconductor die is attached to the substrate and electrically connected to various electrical connections of the substrate using ball attach or wire-bonding techniques. The wire bonds and the die are typically encapsulated with a protective layer of encapsulant. Such packages and the methods of their 

In [68]:
print("\n")
print("=" * 80)
print("PATENTS USED")
print("=" * 80)

for i, row in results.iterrows():

    print(
        f"{i + 1}. "
        f"{row['document_id']} | "
        f"{row['category']} | "
        f"{row['section']} | "
        f"similarity={row['similarity']:.4f}"
    )



PATENTS USED
1. 01ea3f6e9244856597cb | advanced_packaging | description | similarity=0.7394
2. f2ada019c57f2835065e | advanced_packaging | description | similarity=0.7325
3. a969f5fbe709d62ce477 | advanced_packaging | description | similarity=0.7278
4. b8ead524636a26d39526 | advanced_packaging | description | similarity=0.7261
5. a412f1b3ea8b5836df4b | advanced_packaging | description | similarity=0.7229


In [69]:
# STEP 18A — Build Patent-Level Metadata

patent_metadata = []

for file_path in patent_files:

    try:
        parsed = parse_patent_file(file_path)

        patent_metadata.append({
            "document_id": parsed["document_id"],
            "category": parsed["category"],
            "cpc_section": parsed["cpc_section"],
            "source": parsed["source"],
            "abstract": parsed["abstract"],
            "claims": parsed["claims"],
            "file_path": parsed["file_path"],
        })

    except Exception as e:
        print(f"Failed to parse {file_path}: {e}")


patent_metadata_df = pd.DataFrame(patent_metadata)

print("Patent metadata loaded")
print("Patents:", len(patent_metadata_df))
print("Columns:")
print(patent_metadata_df.columns.tolist())

Patent metadata loaded
Patents: 500
Columns:
['document_id', 'category', 'cpc_section', 'source', 'abstract', 'claims', 'file_path']


In [70]:
# STEP 18B — Check Claims Coverage

claims_available = patent_metadata_df["claims"].fillna("").str.strip().ne("")

print("=" * 80)
print("PATENT CLAIMS COVERAGE")
print("=" * 80)

print("Total patents :", len(patent_metadata_df))
print("With claims   :", claims_available.sum())
print("Without claims:", (~claims_available).sum())

print(
    "Coverage      :",
    round(claims_available.mean() * 100, 2),
    "%"
)

PATENT CLAIMS COVERAGE
Total patents : 500
With claims   : 0
Without claims: 500
Coverage      : 0.0 %


In [71]:
# STEP 18C — Enrich Retrieved Results with Patent Evidence

def enrich_retrieval_results(results):
    """
    Add patent-level metadata, abstracts, and claims
    to semantic retrieval results.
    """

    enriched = results.copy()

    metadata_lookup = patent_metadata_df.set_index(
        "document_id"
    )

    enriched["abstract"] = ""
    enriched["claims"] = ""

    for idx, row in enriched.iterrows():

        patent_id = row["document_id"]

        if patent_id in metadata_lookup.index:

            patent_info = metadata_lookup.loc[patent_id]

            enriched.at[idx, "abstract"] = patent_info["abstract"]
            enriched.at[idx, "claims"] = patent_info["claims"]

    return enriched

In [72]:
enriched_results = enrich_retrieval_results(results)

print(enriched_results.columns.tolist())

['chunk_id', 'document_id', 'category', 'cpc_section', 'source', 'section', 'text', 'file_path', 'similarity', 'abstract', 'claims']


In [73]:
# STEP 18D — Inspect Patent Evidence

for i, row in enriched_results.iterrows():

    print("=" * 80)
    print(f"PATENT {i + 1}")
    print("=" * 80)

    print("Patent ID :", row["document_id"])
    print("Category  :", row["category"])
    print("CPC       :", row["cpc_section"])
    print("Source    :", row["source"])
    print("Similarity:", round(float(row["similarity"]), 4))

    print("\nABSTRACT")
    print("-" * 80)
    print(row["abstract"][:1500])

    print("\nCLAIMS")
    print("-" * 80)

    if row["claims"]:
        print(row["claims"][:2500])
    else:
        print("No claims available.")

    print("\nRETRIEVED DESCRIPTION CHUNK")
    print("-" * 80)
    print(row["text"][:1500])

    print()

PATENT 1
Patent ID : 01ea3f6e9244856597cb
Category  : advanced_packaging
CPC       : H
Source    : BIGPATENT
Similarity: 0.7394

ABSTRACT
--------------------------------------------------------------------------------
Embodiments of the invention include a semiconductor integrated circuit package that includes a substrate having an integrated circuit die attached thereto. The package includes a ESD shield attached to the substrate. The ESD shield configured to increase the ESD hardness of the package. The ESD shield can further serve to stiffen the package to prevent warping and operate as a heat spreader.

CLAIMS
--------------------------------------------------------------------------------
No claims available.

RETRIEVED DESCRIPTION CHUNK
--------------------------------------------------------------------------------
TECHNICAL FIELD 
 The invention described herein relates generally to semiconductor device packaging. In particular, the invention relates to a method and package fo

In [74]:
# STEP 18E — Build Evidence-Aware RAG Context

def build_evidence_context(results):
    """
    Build structured evidence context for the LLM.

    Each result contains:
    - patent metadata
    - abstract
    - claims
    - retrieved description chunk
    - similarity score
    """

    context_parts = []

    for i, row in results.iterrows():

        abstract = row.get("abstract", "")
        claims = row.get("claims", "")

        if not abstract:
            abstract = "No abstract available."

        if not claims:
            claims = "No claims available."

        context_parts.append(
            f"""
PATENT EVIDENCE {i + 1}
======================

Patent ID:
{row["document_id"]}

Category:
{row["category"]}

CPC Section:
{row["cpc_section"]}

Source:
{row["source"]}

Semantic Similarity:
{row["similarity"]:.4f}


ABSTRACT
--------
{abstract[:2000]}


CLAIMS
------
{claims[:3000]}


RETRIEVED PATENT TEXT
---------------------
{row["text"][:3000]}

"""
        )

    return "\n".join(context_parts)

In [78]:
# STEP 19A — Citation-Aware RAG Prompt

def build_citation_rag_prompt(question, retrieved_context):
    """
    Build a RAG prompt that requires patent-level citations.
    """

    prompt = f"""
You are a semiconductor patent research assistant.

Answer the user's question using ONLY the retrieved patent
evidence provided below.

IMPORTANT RULES
===============

1. Do not use outside knowledge.
2. Do not invent patent IDs.
3. Every technical statement must be supported by the
   retrieved patent evidence.
4. Cite the patent ID immediately after the statement it supports.
5. Use this citation format:

   [Patent: DOCUMENT_ID]

6. If multiple patents support a statement, cite them all:

   [Patent: ID1, ID2]

7. If the retrieved evidence is insufficient, say:

   "The retrieved patent evidence is insufficient to answer this."

8. Do not claim that something is covered by a patent claim,
   because this corpus does not contain claims.
9. Do not provide legal opinions about patent validity,
   infringement, novelty, or claim coverage.

RETRIEVED PATENT EVIDENCE
=========================

{retrieved_context}

=========================

USER QUESTION
=============

{question}

=========================

ANSWER REQUIREMENTS
===================

Provide:

1. A concise answer.
2. The main technical approaches found in the retrieved patents.
3. A patent citation after each relevant technical statement.
4. A short Sources section listing the patent IDs used.

ANSWER
======
"""

    return prompt

In [79]:
# STEP 19B — Citation-Aware Evidence Context

def build_citation_context(results):
    """
    Build evidence context where each patent is clearly
    identified for citation.
    """

    context_parts = []

    for i, row in results.iterrows():

        abstract = row.get("abstract", "")

        if not abstract:
            abstract = "No abstract available."

        context_parts.append(
            f"""
SOURCE {i + 1}
============

PATENT ID:
{row["document_id"]}

CATEGORY:
{row["category"]}

CPC SECTION:
{row["cpc_section"]}

SOURCE:
{row["source"]}

SIMILARITY:
{row["similarity"]:.4f}

ABSTRACT:
{abstract[:2000]}

RETRIEVED DESCRIPTION:
{row["text"][:3500]}

END SOURCE
==========
"""
        )

    return "\n".join(context_parts)

In [80]:
# STEP 19C — Citation-Aware RAG Query

question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

results = retrieve_patents(
    question,
    top_k=5
)

enriched_results = enrich_retrieval_results(
    results
)

context = build_citation_context(
    enriched_results
)

prompt = build_citation_rag_prompt(
    question,
    context
)

answer = ask_ollama(prompt)

print("=" * 80)
print("CITATION-AWARE RAG ANSWER")
print("=" * 80)
print(answer)

CITATION-AWARE RAG ANSWER
The retrieved patent evidence mentions techniques to reduce impedance in semiconductor packaging, primarily through the use of a terminating resistor connected to ground to mitigate plating stub effects. This resistor helps control signal reflections in high-speed interconnects, thereby reducing impedance issues caused by plating stubs. [Patent: a969f5fbe709d62ce477]

**Sources**  
a969f5fbe709d62ce477, f2ada019c57f2835065e, 01ea3f6e9244856597cb, b8ead524636a26d39526, a412f1b3ea8b5836df4b


In [81]:
# STEP 19D — Show Retrieved Patent Sources

print("\n")
print("=" * 80)
print("PATENTS USED")
print("=" * 80)

for i, row in enriched_results.iterrows():

    print(
        f"{i + 1}. "
        f"{row['document_id']} | "
        f"{row['category']} | "
        f"{row['section']} | "
        f"similarity={row['similarity']:.4f}"
    )



PATENTS USED
1. 01ea3f6e9244856597cb | advanced_packaging | description | similarity=0.7394
2. f2ada019c57f2835065e | advanced_packaging | description | similarity=0.7325
3. a969f5fbe709d62ce477 | advanced_packaging | description | similarity=0.7278
4. b8ead524636a26d39526 | advanced_packaging | description | similarity=0.7261
5. a412f1b3ea8b5836df4b | advanced_packaging | description | similarity=0.7229


In [82]:
# STEP 20A — Two-Stage Retrieval

def retrieve_patents_two_stage(
    query,
    candidate_k=20,
    top_patents=5,
    chunks_per_patent=2,
):
    """
    Two-stage patent retrieval.

    Stage 1:
        Retrieve the most similar chunks.

    Stage 2:
        Select the strongest distinct patents and retain
        multiple relevant chunks from each patent.
    """

    query_embedding = np.array(
        embed_text(query),
        dtype=np.float32
    )

    query_norm = np.linalg.norm(query_embedding)

    query_embedding = (
        query_embedding / max(query_norm, 1e-12)
    )

    similarities = (
        embedding_matrix_normalized @ query_embedding
    )

    # --------------------------------------------------------
    # Stage 1 — Retrieve broad candidate pool
    # --------------------------------------------------------

    ranked_indices = np.argsort(similarities)[::-1]

    candidate_indices = ranked_indices[:candidate_k]

    candidates = chunks_df.iloc[candidate_indices].copy()

    candidates["similarity"] = similarities[candidate_indices]

    candidates = candidates.sort_values(
        "similarity",
        ascending=False
    )

    # --------------------------------------------------------
    # Stage 2 — Select distinct patents
    # --------------------------------------------------------

    selected_patents = []

    for patent_id in candidates["document_id"]:

        if patent_id not in selected_patents:

            selected_patents.append(patent_id)

        if len(selected_patents) >= top_patents:
            break

    # --------------------------------------------------------
    # Keep multiple chunks from selected patents
    # --------------------------------------------------------

    final_results = []

    for patent_id in selected_patents:

        patent_chunks = candidates[
            candidates["document_id"] == patent_id
        ].head(chunks_per_patent)

        final_results.append(patent_chunks)

    if not final_results:
        return pd.DataFrame(
            columns=list(chunks_df.columns) + ["similarity"]
        )

    results = pd.concat(
        final_results,
        ignore_index=True
    )

    # Highest similarity first
    results = results.sort_values(
        "similarity",
        ascending=False
    ).reset_index(drop=True)

    return results

In [83]:
# STEP 20B — Test Two-Stage Retrieval

question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

results = retrieve_patents_two_stage(
    question,
    candidate_k=20,
    top_patents=5,
    chunks_per_patent=2,
)

print("=" * 80)
print("TWO-STAGE RETRIEVAL")
print("=" * 80)

print("Results:", len(results))
print(
    "Unique patents:",
    results["document_id"].nunique()
)

print()

for i, row in results.iterrows():

    print(
        f"{i + 1}. "
        f"{row['document_id']} | "
        f"{row['section']} | "
        f"similarity={row['similarity']:.4f}"
    )

TWO-STAGE RETRIEVAL
Results: 7
Unique patents: 5

1. 01ea3f6e9244856597cb | description | similarity=0.7394
2. f2ada019c57f2835065e | description | similarity=0.7325
3. a969f5fbe709d62ce477 | description | similarity=0.7278
4. 01ea3f6e9244856597cb | description | similarity=0.7273
5. b8ead524636a26d39526 | description | similarity=0.7261
6. a412f1b3ea8b5836df4b | description | similarity=0.7229
7. a969f5fbe709d62ce477 | abstract | similarity=0.7220


In [84]:
# STEP 20C — Enrich Two-Stage Results

enriched_results = enrich_retrieval_results(
    results
)

print(enriched_results.columns.tolist())

['chunk_id', 'document_id', 'category', 'cpc_section', 'source', 'section', 'text', 'file_path', 'similarity', 'abstract', 'claims']


In [85]:
# STEP 20D — Build Multi-Chunk Evidence Context

def build_citation_context(results):

    context_parts = []

    grouped = results.groupby("document_id")

    source_number = 1

    for patent_id, patent_group in grouped:

        first_row = patent_group.iloc[0]

        abstract = first_row.get(
            "abstract",
            ""
        )

        if not abstract:
            abstract = "No abstract available."

        chunks = []

        for j, (_, row) in enumerate(
            patent_group.iterrows(),
            start=1
        ):

            chunks.append(
                f"""
RELEVANT TEXT CHUNK {j}
Similarity: {row["similarity"]:.4f}

{row["text"]}
"""
            )

        context_parts.append(
            f"""
SOURCE {source_number}
=====================

PATENT ID:
{patent_id}

CATEGORY:
{first_row["category"]}

CPC SECTION:
{first_row["cpc_section"]}

SOURCE:
{first_row["source"]}

ABSTRACT:
{abstract[:2000]}

RELEVANT PATENT TEXT:
{"".join(chunks)}

END SOURCE
==========
"""
        )

        source_number += 1

    return "\n".join(context_parts)

In [86]:
# STEP 20E — Two-Stage Citation-Aware RAG

question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

results = retrieve_patents_two_stage(
    question,
    candidate_k=20,
    top_patents=5,
    chunks_per_patent=2,
)

enriched_results = enrich_retrieval_results(
    results
)

context = build_citation_context(
    enriched_results
)

prompt = build_citation_rag_prompt(
    question,
    context
)

answer = ask_ollama(prompt)

print("=" * 80)
print("TWO-STAGE RAG ANSWER")
print("=" * 80)

print(answer)

TWO-STAGE RAG ANSWER
**Answer:**  
Techniques to reduce impedance in semiconductor packaging include:  
1. **Using an ESD shield** to increase package impedance, enhancing resistance to electrostatic discharge (ESD) failures. This approach improves impedance management for ESD protection. (Patent: 01ea3f6e9244856597cb)  
2. **Employing terminating resistors** connected to ground to mitigate plating stub effects in high-speed signal paths. This technique reduces impedance mismatches and minimizes signal reflections. (Patent: a969f5fbe709d62ce477)  

**Sources:**  
- 01ea3f6e9244856597cb  
- a969f5fbe709d62ce477


In [87]:
# STEP 20F — Display Patent Sources

print("\n")
print("=" * 80)
print("PATENTS USED")
print("=" * 80)

for patent_id, group in enriched_results.groupby(
    "document_id"
):

    best_similarity = group["similarity"].max()

    print(
        f"{patent_id} | "
        f"chunks={len(group)} | "
        f"best_similarity={best_similarity:.4f}"
    )



PATENTS USED
01ea3f6e9244856597cb | chunks=2 | best_similarity=0.7394
a412f1b3ea8b5836df4b | chunks=1 | best_similarity=0.7229
a969f5fbe709d62ce477 | chunks=2 | best_similarity=0.7278
b8ead524636a26d39526 | chunks=1 | best_similarity=0.7261
f2ada019c57f2835065e | chunks=1 | best_similarity=0.7325


In [88]:
# STEP 21A — Retrieval Evaluation Questions

evaluation_questions = [
    "What techniques are used to reduce impedance in semiconductor packaging?",
    
    "What methods are used for semiconductor wafer defect detection?",
    
    "How are thermal management problems addressed in semiconductor packages?",
    
    "What techniques are used to improve semiconductor interconnect reliability?",
    
    "What methods are used to improve semiconductor manufacturing yield?",
    
    "How are semiconductor devices packaged to reduce signal loss?",
    
    "What techniques are used for wafer-level packaging?",
    
    "How are semiconductor manufacturing defects identified?",
    
    "What methods are used to improve heat dissipation in semiconductor devices?",
    
    "What approaches are used to improve electrical performance in advanced semiconductor packaging?",
]

print("Evaluation questions:", len(evaluation_questions))

Evaluation questions: 10


In [89]:
# STEP 21B — Run Retrieval Evaluation

evaluation_results = []

for question in evaluation_questions:

    results = retrieve_patents_two_stage(
        question,
        candidate_k=20,
        top_patents=5,
        chunks_per_patent=2,
    )

    unique_patents = results["document_id"].unique()

    for rank, patent_id in enumerate(unique_patents, start=1):

        patent_rows = results[
            results["document_id"] == patent_id
        ]

        best_similarity = patent_rows["similarity"].max()

        category = patent_rows.iloc[0]["category"]

        evaluation_results.append({
            "question": question,
            "rank": rank,
            "document_id": patent_id,
            "category": category,
            "similarity": best_similarity,
        })

evaluation_df = pd.DataFrame(evaluation_results)

print(
    "Evaluation completed:",
    len(evaluation_df),
    "patent/question results"
)

Evaluation completed: 50 patent/question results


In [90]:
# STEP 21C — Inspect Retrieval Results

for question in evaluation_questions:

    print("=" * 100)
    print("QUESTION")
    print(question)
    print("=" * 100)

    subset = evaluation_df[
        evaluation_df["question"] == question
    ]

    for _, row in subset.iterrows():

        print(
            f"{int(row['rank'])}. "
            f"{row['document_id']} | "
            f"{row['category']} | "
            f"similarity={row['similarity']:.4f}"
        )

    print()

QUESTION
What techniques are used to reduce impedance in semiconductor packaging?
1. 01ea3f6e9244856597cb | advanced_packaging | similarity=0.7394
2. f2ada019c57f2835065e | advanced_packaging | similarity=0.7325
3. a969f5fbe709d62ce477 | advanced_packaging | similarity=0.7278
4. b8ead524636a26d39526 | advanced_packaging | similarity=0.7261
5. a412f1b3ea8b5836df4b | advanced_packaging | similarity=0.7229

QUESTION
What methods are used for semiconductor wafer defect detection?
1. patent_eccc29daa65f9d0d | transistor_device_technology | similarity=0.7784
2. patent_678e9b7f28681857 | semiconductor_manufacturing | similarity=0.7644
3. patent_1bc765dc24dc5424 | semiconductor_manufacturing | similarity=0.7294
4. patent_5857feba838ea63a | advanced_packaging | similarity=0.7079
5. be1f8e667484e2e2008a | advanced_packaging | similarity=0.7075

QUESTION
How are thermal management problems addressed in semiconductor packages?
1. 6001024cc3df05a76add | advanced_packaging | similarity=0.7370
2. 4f2

In [91]:
# STEP 21D — Expected Categories

expected_categories = {
    "What techniques are used to reduce impedance in semiconductor packaging?":
        ["advanced_packaging"],

    "What methods are used for semiconductor wafer defect detection?":
        ["semiconductor_manufacturing"],

    "How are thermal management problems addressed in semiconductor packages?":
        ["advanced_packaging"],

    "What techniques are used to improve semiconductor interconnect reliability?":
        ["advanced_packaging", "semiconductor_manufacturing"],

    "What methods are used to improve semiconductor manufacturing yield?":
        ["semiconductor_manufacturing"],

    "How are semiconductor devices packaged to reduce signal loss?":
        ["advanced_packaging"],

    "What techniques are used for wafer-level packaging?":
        ["advanced_packaging"],

    "How are semiconductor manufacturing defects identified?":
        ["semiconductor_manufacturing"],

    "What methods are used to improve heat dissipation in semiconductor devices?":
        ["advanced_packaging"],

    "What approaches are used to improve electrical performance in advanced semiconductor packaging?":
        ["advanced_packaging"],
}

In [92]:
# STEP 21E — Calculate Category Hit Rate

evaluation_summary = []

for question in evaluation_questions:

    expected = expected_categories[question]

    subset = evaluation_df[
        evaluation_df["question"] == question
    ]

    retrieved_categories = set(
        subset["category"].dropna()
    )

    hit = any(
        category in retrieved_categories
        for category in expected
    )

    top1_category = (
        subset.iloc[0]["category"]
        if len(subset) > 0
        else None
    )

    evaluation_summary.append({
        "question": question,
        "expected_categories": ", ".join(expected),
        "top1_category": top1_category,
        "top5_categories": ", ".join(
            sorted(retrieved_categories)
        ),
        "category_hit": hit,
    })

evaluation_summary_df = pd.DataFrame(
    evaluation_summary
)

print(evaluation_summary_df)

                                            question  \
0  What techniques are used to reduce impedance i...   
1  What methods are used for semiconductor wafer ...   
2  How are thermal management problems addressed ...   
3  What techniques are used to improve semiconduc...   
4  What methods are used to improve semiconductor...   
5  How are semiconductor devices packaged to redu...   
6  What techniques are used for wafer-level packa...   
7  How are semiconductor manufacturing defects id...   
8  What methods are used to improve heat dissipat...   
9  What approaches are used to improve electrical...   

                               expected_categories  \
0                               advanced_packaging   
1                      semiconductor_manufacturing   
2                               advanced_packaging   
3  advanced_packaging, semiconductor_manufacturing   
4                      semiconductor_manufacturing   
5                               advanced_packaging   
6    

In [93]:
# STEP 21F — Overall Retrieval Score

category_hit_rate = (
    evaluation_summary_df["category_hit"].mean()
)

print("=" * 80)
print("RETRIEVAL EVALUATION")
print("=" * 80)

print(
    "Category hit rate:",
    round(category_hit_rate * 100, 2),
    "%"
)

RETRIEVAL EVALUATION
Category hit rate: 100.0 %


In [102]:
# STEP 22 — Lightweight Reranking Prompt

def build_reranking_prompt(question, candidates):
    """
    Build a compact reranking prompt for Qwen3.
    """

    candidate_text = []

    for i, row in candidates.iterrows():

        candidate_text.append(
            f"""
[{i}]
Patent: {row["document_id"]}
Text: {row["text"][:900]}
"""
        )

    candidates_text = "\n".join(candidate_text)

    prompt = f"""
Rank these patent passages for the question below.

QUESTION:
{question}

CANDIDATES:
{candidates_text}

Return ONLY the candidate numbers from most relevant
to least relevant.

Example:
3, 1, 0, 4, 2
"""

    return prompt

In [103]:
# STEP 22B — Parse Reranking Output

import re


def parse_ranking(response, candidate_count):
    """
    Extract candidate numbers from the Qwen3 response.
    """

    numbers = re.findall(
        r"\b\d+\b",
        response
    )

    ranking = []

    for number in numbers:

        index = int(number)

        if 0 <= index < candidate_count:
            if index not in ranking:
                ranking.append(index)

    return ranking

In [104]:
# STEP 22C — Qwen3 Reranker

def rerank_candidates(question, candidates):
    """
    Rerank semantic-search candidates using Qwen3.
    """

    if candidates.empty:
        return candidates

    prompt = build_reranking_prompt(
        question,
        candidates
    )

    response = ask_ollama(prompt)

    ranking = parse_ranking(
        response,
        len(candidates)
    )

    # If Qwen3 produced an invalid ranking,
    # preserve the original similarity ordering.
    if not ranking:
        return candidates.copy()

    ranked_rows = []

    for index in ranking:
        ranked_rows.append(
            candidates.iloc[index]
        )

    # Add candidates that Qwen3 failed to mention
    for index in range(len(candidates)):

        if index not in ranking:
            ranked_rows.append(
                candidates.iloc[index]
            )

    reranked = pd.DataFrame(
        ranked_rows
    ).reset_index(drop=True)

    reranked["rerank_position"] = (
        range(1, len(reranked) + 1)
    )

    return reranked

In [105]:
# STEP 22D — Test Reranking

question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

candidate_results = retrieve_patents_two_stage(
    question,
    candidate_k=20,
    top_patents=5,
    chunks_per_patent=2,
)

reranked_results = rerank_candidates(
    question,
    candidate_results
)

print("=" * 80)
print("RERANKED RESULTS")
print("=" * 80)

for i, row in reranked_results.iterrows():

    print(
        f"{i + 1}. "
        f"{row['document_id']} | "
        f"similarity={row['similarity']:.4f}"
    )

RERANKED RESULTS
1. a969f5fbe709d62ce477 | similarity=0.7220
2. a969f5fbe709d62ce477 | similarity=0.7278
3. a412f1b3ea8b5836df4b | similarity=0.7229
4. f2ada019c57f2835065e | similarity=0.7325
5. b8ead524636a26d39526 | similarity=0.7261
6. 01ea3f6e9244856597cb | similarity=0.7394
7. 01ea3f6e9244856597cb | similarity=0.7273


In [106]:
# STEP 23A — Complete Reranked RAG Pipeline

def run_reranked_rag(
    question,
    candidate_k=20,
    top_patents=5,
    chunks_per_patent=2,
):
    """
    Complete patent RAG pipeline:

    1. Semantic candidate retrieval
    2. Patent-level selection
    3. Qwen3 reranking
    4. Patent evidence enrichment
    5. Citation-aware context construction
    6. Grounded answer generation
    """

    # --------------------------------------------------------
    # Stage 1 — Retrieve candidates
    # --------------------------------------------------------

    candidate_results = retrieve_patents_two_stage(
        question,
        candidate_k=candidate_k,
        top_patents=top_patents,
        chunks_per_patent=chunks_per_patent,
    )

    if candidate_results.empty:
        return {
            "question": question,
            "answer": "No relevant patent evidence was retrieved.",
            "results": candidate_results,
        }

    # --------------------------------------------------------
    # Stage 2 — Rerank candidates
    # --------------------------------------------------------

    reranked_results = rerank_candidates(
        question,
        candidate_results,
    )

    # --------------------------------------------------------
    # Stage 3 — Enrich with patent metadata
    # --------------------------------------------------------

    enriched_results = enrich_retrieval_results(
        reranked_results
    )

    # --------------------------------------------------------
    # Stage 4 — Build evidence context
    # --------------------------------------------------------

    context = build_citation_context(
        enriched_results
    )

    # --------------------------------------------------------
    # Stage 5 — Build grounded prompt
    # --------------------------------------------------------

    prompt = build_citation_rag_prompt(
        question,
        context
    )

    # --------------------------------------------------------
    # Stage 6 — Generate answer
    # --------------------------------------------------------

    answer = ask_ollama(prompt)

    return {
        "question": question,
        "answer": answer,
        "results": enriched_results,
        "context": context,
        "prompt": prompt,
    }

In [107]:
# STEP 23B — Run Complete Reranked RAG

question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

rag_result = run_reranked_rag(
    question
)

print("=" * 80)
print("RERANKED RAG ANSWER")
print("=" * 80)

print(rag_result["answer"])

RERANKED RAG ANSWER
The retrieved patent evidence indicates that techniques to reduce impedance in semiconductor packaging include the use of a terminating resistor connected to an electrical ground to mitigate plating stub effects. This resistor is electrically connected to the first end portion of a plating stub, which terminates near the peripheral edge of the interposer substrate, thereby managing impedance in high-speed signal transmission. [Patent: a969f5fbe709d62ce477]  

**Sources**  
- a969f5fbe709d62ce477


In [108]:
# STEP 23C — Display Reranked Evidence

results = rag_result["results"]

print("\n")
print("=" * 80)
print("RERANKED PATENT EVIDENCE")
print("=" * 80)

for i, row in results.iterrows():

    print(
        f"{i + 1}. "
        f"{row['document_id']} | "
        f"similarity={row['similarity']:.4f} | "
        f"rerank_position={row['rerank_position']}"
    )

    print(
        f"   Category: {row['category']}"
    )

    print(
        f"   Section: {row['section']}"
    )

    print()



RERANKED PATENT EVIDENCE
1. a969f5fbe709d62ce477 | similarity=0.7220 | rerank_position=1
   Category: advanced_packaging
   Section: abstract

2. a412f1b3ea8b5836df4b | similarity=0.7229 | rerank_position=2
   Category: advanced_packaging
   Section: description

3. a969f5fbe709d62ce477 | similarity=0.7278 | rerank_position=3
   Category: advanced_packaging
   Section: description

4. b8ead524636a26d39526 | similarity=0.7261 | rerank_position=4
   Category: advanced_packaging
   Section: description

5. f2ada019c57f2835065e | similarity=0.7325 | rerank_position=5
   Category: advanced_packaging
   Section: description

6. 01ea3f6e9244856597cb | similarity=0.7394 | rerank_position=6
   Category: advanced_packaging
   Section: description

7. 01ea3f6e9244856597cb | similarity=0.7273 | rerank_position=7
   Category: advanced_packaging
   Section: description



In [109]:
# STEP 23D — Inspect Evidence Used by the LLM

for i, row in results.iterrows():

    print("=" * 80)
    print(
        f"EVIDENCE {i + 1} — "
        f"{row['document_id']}"
    )
    print("=" * 80)

    print(
        "Similarity:",
        round(float(row["similarity"]), 4)
    )

    print(
        "Rerank position:",
        row["rerank_position"]
    )

    print("\nPatent text:")
    print(row["text"])

    print()

EVIDENCE 1 — a969f5fbe709d62ce477
Similarity: 0.722
Rerank position: 1

Patent text:
Packages and methods for mitigating plating stub effects. The semiconductor package includes an interposer substrate having a first side, a second side, a peripheral edge connecting the first side with the second side, a signal line on the first side, and an electrode pad on the first side. A semiconductor element is mounted on the first side of the interposer substrate. The semiconductor element is connected with the electrode pad by the signal line. A terminating resistor is mounted on the interposer substrate. A plating stub, which is located on the interposer substrate, has a first end portion that terminates near the peripheral edge of the interposer substrate and a second end portion that is electrically connected to the electrode. The first end portion is electrically connected through the terminating resistor to an electrical ground.

EVIDENCE 2 — a412f1b3ea8b5836df4b
Similarity: 0.7229
Rerank 

In [110]:
# STEP 23E — Multi-Question RAG Test

test_questions = [
    "What techniques are used to reduce impedance in semiconductor packaging?",
    
    "What methods are used for semiconductor wafer defect detection?",
    
    "How are thermal management problems addressed in semiconductor packages?",
    
    "What techniques are used to improve semiconductor interconnect reliability?",
    
    "What methods are used to improve semiconductor manufacturing yield?",
]

In [112]:
results = result["results"]

print("=" * 80)
print("TOP PATENTS")
print("=" * 80)

if results.empty:
    print("No patents retrieved.")
else:
    top_patents = (
        results
        .groupby("document_id", sort=False)
        .first()
        .reset_index()
    )

    for _, row in top_patents.iterrows():
        print(
            f"- {row['document_id']} "
            f"(similarity={row['similarity']:.4f})"
        )

print("\n")
rag_test_results.append(result)

TOP PATENTS
- a969f5fbe709d62ce477 (similarity=0.7220)
- a412f1b3ea8b5836df4b (similarity=0.7229)
- b8ead524636a26d39526 (similarity=0.7261)
- f2ada019c57f2835065e (similarity=0.7325)
- 01ea3f6e9244856597cb (similarity=0.7394)




In [119]:
def run_reranked_rag(
    question,
    candidate_k=20,
    top_patents=5,
    chunks_per_patent=2,
    rerank_k=10,
):
    """
    Complete semiconductor patent RAG pipeline:

    1. Vector retrieval
    2. Patent-level candidate selection
    3. Qwen3 reranking
    4. Patent metadata enrichment
    5. Citation-aware answer generation
    """

    # --------------------------------------------------------
    # 1. Retrieve candidates
    # --------------------------------------------------------

    candidate_results = retrieve_patents_two_stage(
        question,
        candidate_k=candidate_k,
        top_patents=top_patents,
        chunks_per_patent=chunks_per_patent,
    )

    if candidate_results.empty:
        return {
            "question": question,
            "answer": (
                "No relevant patent evidence was retrieved."
            ),
            "results": candidate_results,
        }

    # --------------------------------------------------------
    # 2. Rerank with Qwen3
    # --------------------------------------------------------

    reranked_results = rerank_candidates(
        question,
        candidate_results,
        rerank_k=rerank_k,
    )

    # --------------------------------------------------------
    # 3. Enrich with patent-level metadata
    # --------------------------------------------------------

    enriched_results = enrich_retrieval_results(
        reranked_results
    )

    # --------------------------------------------------------
    # 4. Build citation-aware context
    # --------------------------------------------------------

    context = build_citation_context(
        enriched_results
    )

    # --------------------------------------------------------
    # 5. Generate grounded answer
    # --------------------------------------------------------

    prompt = build_citation_rag_prompt(
        question,
        context
    )

    answer = ask_ollama(prompt)

    return {
        "question": question,
        "answer": answer,
        "results": enriched_results,
        "context": context,
        "prompt": prompt,
    }

In [121]:
def run_reranked_rag(
    question,
    candidate_k=20,
    top_patents=5,
    chunks_per_patent=2,
    rerank_k=10,
):
    """
    Complete semiconductor patent RAG pipeline:

    1. Vector retrieval
    2. Patent-level candidate selection
    3. Qwen3 reranking
    4. Patent metadata enrichment
    5. Citation-aware answer generation
    """

    # --------------------------------------------------------
    # 1. Retrieve candidates
    # --------------------------------------------------------

    candidate_results = retrieve_patents_two_stage(
        question,
        candidate_k=candidate_k,
        top_patents=top_patents,
        chunks_per_patent=chunks_per_patent,
    )

    if candidate_results.empty:
        return {
            "question": question,
            "answer": (
                "No relevant patent evidence was retrieved."
            ),
            "results": candidate_results,
        }

    # --------------------------------------------------------
    # 2. Rerank with Qwen3
    # --------------------------------------------------------

    reranked_results = rerank_candidates(
        question,
        candidate_results,
        rerank_k=rerank_k,
    )

    # --------------------------------------------------------
    # 3. Enrich with patent-level metadata
    # --------------------------------------------------------

    enriched_results = enrich_retrieval_results(
        reranked_results
    )

    # --------------------------------------------------------
    # 4. Build citation-aware context
    # --------------------------------------------------------

    context = build_citation_context(
        enriched_results
    )

    # --------------------------------------------------------
    # 5. Generate grounded answer
    # --------------------------------------------------------

    prompt = build_citation_rag_prompt(
        question,
        context
    )

    answer = ask_ollama(prompt)

    return {
        "question": question,
        "answer": answer,
        "results": enriched_results,
        "context": context,
        "prompt": prompt,
    }

In [122]:
question = (
    "What techniques are used to reduce impedance "
    "in semiconductor packaging?"
)

candidate_results = retrieve_patents_two_stage(
    question,
    candidate_k=20,
    top_patents=5,
    chunks_per_patent=2,
)

print("=" * 80)
print("VECTOR CANDIDATES")
print("=" * 80)

for i, row in candidate_results.iterrows():
    print(
        f"{i}. {row['document_id']} | "
        f"{row['section']} | "
        f"similarity={row['similarity']:.4f}"
    )

reranked_results = rerank_candidates(
    question,
    candidate_results,
    rerank_k=10,
)

print("\n")
print("=" * 80)
print("RERANKED RESULTS")
print("=" * 80)

for i, row in reranked_results.iterrows():
    print(
        f"{i + 1}. {row['document_id']} | "
        f"similarity={row['similarity']:.4f} | "
        f"rerank={row['rerank_position']}"
    )

VECTOR CANDIDATES
0. 01ea3f6e9244856597cb | description | similarity=0.7394
1. f2ada019c57f2835065e | description | similarity=0.7325
2. a969f5fbe709d62ce477 | description | similarity=0.7278
3. 01ea3f6e9244856597cb | description | similarity=0.7273
4. b8ead524636a26d39526 | description | similarity=0.7261
5. a412f1b3ea8b5836df4b | description | similarity=0.7229
6. a969f5fbe709d62ce477 | abstract | similarity=0.7220


TypeError: rerank_candidates() got an unexpected keyword argument 'rerank_k'